# Tax Data Foundation: Raw to Return-Ready

**Lab 1 of 2** · 57 min core + 28 min optional · Beginner → Intermediate

Three incompatible ERP extracts, 2.4 million ledger lines, one governed GST position — and an audit trail you can reproduce on demand.

Work through the cells top to bottom. Every SQL cell is ready to run - there is nothing to copy or paste.

**Before you start**, run `gsi_tax_lab0_setup.ipynb` if you have not already. Set this notebook's role to `SYSADMIN` and the warehouse to `GSI_TAX_WH`. Step 1 verifies the foundation is present before you go any further.

> Architecture, the business process behind it and how the notebooks fit together: **`gsi_tax_workshop_lab1_foundation.html`**. You do not need it open to work through this notebook - everything you need to run and interpret each step is here.

---

> ⚠️ **Synthetic data**
>
> All data in this workshop is synthetically generated for training purposes. Entity names, ABNs, suppliers and transactions are fictitious. Tax treatments shown are simplified for illustration and are not tax advice.

---

## Step 1 of 8 — Verify the foundation and look around

**Inspect** → Diagnose → Conform → Determine → Reconcile

*~5 min* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-1)

**What the tax practice is up against.** The group has three general ledgers, inherited from a decade of acquisitions. A partner will shortly ask what the group's GST position is. Before you can even attempt that, you need to see what the three systems actually send — because they do not describe the business the same way.

First confirm the setup notebook has run and left you a complete foundation.

In [ ]:
-- Point the session at the workshop, then confirm the foundation is complete.
-- Every STATUS must read OK. If one does not, run gsi_tax_lab0_setup.ipynb.
USE WAREHOUSE GSI_TAX_WH;
USE DATABASE GSI_TAX_WORKSHOP;

SELECT 'Raw tables landed'          AS CHECK_NAME,
       COUNT(*)::VARCHAR            AS FOUND,
       '8'                          AS EXPECTED,
       IFF(COUNT(*) = 8, 'OK', 'run gsi_tax_lab0_setup.ipynb') AS STATUS
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'RAW_DATA'

UNION ALL
SELECT 'GL lines across three ERPs',
       TO_VARCHAR(SUM(N), '999,999,999'), '2,400,000',
       IFF(SUM(N) = 2400000, 'OK', 'run gsi_tax_lab0_setup.ipynb')
FROM (
    SELECT COUNT(*) AS N FROM RAW_DATA.ERP_SAP_GL
    UNION ALL SELECT COUNT(*) FROM RAW_DATA.ERP_ORACLE_GL
    UNION ALL SELECT COUNT(*) FROM RAW_DATA.ERP_LEGACY_AP
)

UNION ALL
SELECT 'Tax rules the team owns',
       (SELECT COUNT(*)::VARCHAR FROM TAX.TAX_RULE_DETERMINATION), '16',
       IFF((SELECT COUNT(*) FROM TAX.TAX_RULE_DETERMINATION) = 16, 'OK', 'run gsi_tax_lab0_setup.ipynb')

UNION ALL
SELECT 'Lodged BAS entity-periods',
       (SELECT COUNT(*)::VARCHAR FROM RAW_DATA.FCT_BAS_LODGED), '532',
       IFF((SELECT COUNT(*) FROM RAW_DATA.FCT_BAS_LODGED) = 532, 'OK', 'run gsi_tax_lab0_setup.ipynb');

**Four rows, every status OK.** 2,400,000 ledger lines are already queryable, and the count came back in about a second because Snowflake answers it from table metadata rather than reading the rows.

### Look at the three shapes side by side

Nine rows, three source systems, everything cast to `VARCHAR` so nothing is silently coerced on the way to your screen. You see exactly what landed.

In [ ]:
-- Three source systems, nine rows, everything cast to VARCHAR so nothing is
-- silently coerced. Read down DATE_AS_LANDED, then across the two amount columns.
-- Each subquery is ordered so the sample is the same rows every time you run it.
SELECT * FROM (
    SELECT 'SAP'    AS SOURCE_SYSTEM,
           BELNR::VARCHAR              AS DOC_REF,
           BUKRS::VARCHAR              AS ENTITY_AS_LANDED,
           BUDAT::VARCHAR              AS DATE_AS_LANDED,
           DMBTR::VARCHAR              AS AMOUNT_1,
           NULL::VARCHAR               AS AMOUNT_2,
           MWSTS::VARCHAR              AS TAX_AS_BOOKED,
           MWSKZ::VARCHAR              AS TAX_CODE
    FROM RAW_DATA.ERP_SAP_GL ORDER BY BELNR LIMIT 3)
UNION ALL
SELECT * FROM (
    SELECT 'ORACLE',
           JE_HEADER_ID::VARCHAR,
           LEDGER_ID::VARCHAR,
           GL_DATE::VARCHAR,
           ENTERED_DR::VARCHAR,
           ENTERED_CR::VARCHAR,
           TAX_AMOUNT::VARCHAR,
           TAX_CLASSIFICATION::VARCHAR
    FROM RAW_DATA.ERP_ORACLE_GL ORDER BY JE_HEADER_ID LIMIT 3)
UNION ALL
SELECT * FROM (
    SELECT 'LEGACY',
           doc_ref::VARCHAR,
           entity_cd::VARCHAR,
           txn_dt::VARCHAR,
           amt_incl_tax::VARCHAR,
           NULL::VARCHAR,
           tax_amt::VARCHAR,
           tax_flag::VARCHAR
    FROM RAW_DATA.ERP_LEGACY_AP ORDER BY doc_ref LIMIT 3)
ORDER BY SOURCE_SYSTEM, DOC_REF;

**Three systems, three incompatible conventions, one business.** `DATE_AS_LANDED` reads `20250528` in SAP, `2024-05-01` in Oracle and `02/02/2024` in the legacy system. Look at the two amount columns: SAP posts one signed figure (row 2 is negative), Oracle fills either `AMOUNT_1` or `AMOUNT_2` and leaves the other empty, and the legacy system posts one unsigned figure that already includes the GST.

*Snowflake: all three raw shapes queried where they landed. No extract, no staging server, no pre-processing step to maintain.*

### The same business concept, three names

| Business concept | ERP_SAP_GL | ERP_ORACLE_GL | ERP_LEGACY_AP |
|---|---|---|---|
| Document reference | `BELNR` | `JE_HEADER_ID` | `doc_ref` |
| Entity identifier | `BUKRS` — `1010` | `LEDGER_ID` — `LED_2001` | `entity_cd` — `AU01` |
| Posting date | `BUDAT` — text, `YYYYMMDD` | `GL_DATE` — native `DATE` | `txn_dt` — text, `DD/MM/YYYY` |
| Account code | `HKONT` — `400001` | `CODE_COMBINATION` — `5100-0001` | `acct` — `AP001` |
| Amount | `DMBTR` — signed, GST-exclusive | `ENTERED_DR` / `ENTERED_CR` — two columns, unsigned | `amt_incl_tax` — GST-inclusive |
| Tax amount | `MWSTS` | `TAX_AMOUNT` | `tax_amt` |
| Tax code | `MWSKZ` — `S1`, `S0`, `IT`, `EX`, `ZF` | `TAX_CLASSIFICATION` — `STANDARD`, `EXEMPT`, `ZERO_RATED`, `OUT_OF_SCOPE` | `tax_flag` — `Y`, `N`, `X` |
| Supplier | `LIFNR` | `VENDOR_ID` | `vendor_ref` |
| Narrative | `SGTXT` | `DESCRIPTION` | `narrative` |

Nothing here has been cleaned, and nothing will be until Step 3. The raw layer holds data exactly as it landed.

---

## Step 2 of 8 — Prove you cannot answer the question

Inspect → **Diagnose** → Conform → Determine → Reconcile

*~8 min* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-2)

**The question the partner actually asks.** *What was the group's GST payable for the last full quarter, by entity?* It is a completely reasonable question, and right now it is unanswerable. This step proves that in three queries rather than cataloguing every flaw in the data.

### The plausible wrong answer

SAP is the biggest of the three systems and the one the group thinks of as *the* general ledger. Answer the question against SAP alone.

In [ ]:
-- The partner's question, answered against SAP alone.
-- GST booked by company code for the last three complete months.
--
-- The window is pinned to the dataset's as-of date rather than CURRENT_DATE().
-- The ledger ends 2026-07-31, so a relative window would silently start losing
-- months as real time moves on, and every figure quoted below would drift.
SELECT
    g.BUKRS                    AS SAP_COMPANY_CODE,
    e.ENTITY_NAME,
    COUNT(*)                   AS GL_LINES_SCANNED,
    ROUND(SUM(g.MWSTS), 2)     AS GST_BOOKED_AUD
FROM RAW_DATA.ERP_SAP_GL g
LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e
       ON e.SAP_BUKRS = g.BUKRS
WHERE TRY_TO_DATE(g.BUDAT, 'YYYYMMDD') >= DATEADD('month', -3, '2026-08-01'::DATE)
  AND TRY_TO_DATE(g.BUDAT, 'YYYYMMDD') <  '2026-08-01'::DATE
GROUP BY 1, 2
ORDER BY GST_BOOKED_AUD DESC;

That returns in about a second: a clean, confident, formatted number per company code, with **94,947 ledger lines scanned** and **$174,806,895** of GST across **12 rows**.

> ✅ **One of your source systems will not open in Excel**
>
> `ERP_SAP_GL` holds 1,200,000 rows. Microsoft Excel stops at 1,048,576 — one of your three source systems, on its own, will not open in the tool your team uses today. That aggregate scanned the whole table on the smallest warehouse Snowflake sells, with no index, no tuning and no preparation of any kind.

> ⚠️ **And it is still the wrong answer**
>
> The group has 14 legal entities. That result has 12 rows, because two entities do not exist in SAP at all — Marrickville Food Co and Nullarbor Freight are booked in Oracle and in the legacy system. Nothing in the result grid says so. A number that is confidently wrong and gives no indication that it is wrong is the single most dangerous artefact in tax reporting.

> 💡 **Profile a column without writing a query**
>
> With those results still on screen, click a column header in the Snowsight results pane. Snowsight shows the value distribution, the null count and the distinct count for that column immediately. If you profile columns by hand in a spreadsheet or a workflow tool today, that is the whole job done in one click.

### So stack the three systems together

The obvious next move is to union the three extracts on the columns that look equivalent. Try it.

> ⚠️ **This cell is meant to fail**
>
> Run it anyway. The error is the teaching point, and the next cell fixes it. You have not broken anything — expect a red cell and read the message.

In [ ]:
-- The obvious next move: stack the three extracts on the columns that
-- look equivalent. Oracle splits the amount across two columns, so the
-- tax code slides into the amount slot. This query FAILS. That is the point.
SELECT BUDAT  AS DOC_DATE, DMBTR        AS AMOUNT, MWSKZ  AS TAX_CODE
FROM RAW_DATA.ERP_SAP_GL
UNION ALL
SELECT GL_DATE, ENTERED_DR, ENTERED_CR
FROM RAW_DATA.ERP_ORACLE_GL
UNION ALL
SELECT txn_dt, amt_incl_tax, tax_flag
FROM RAW_DATA.ERP_LEGACY_AP;

-- Numeric value 'S0' is not recognized

It fails. `Numeric value 'S0' is not recognized`. There is no way to line these three extracts up by position, and there are four separate reasons why.

- **Date types.** One column is a real `DATE`; the other two are text in two different conventions. `20250528` and `02/02/2024` — the two you just saw in Step 1 — cannot be compared, sorted or filtered against each other until both are parsed, and roughly 2% of the legacy dates are not valid dates at all.
- **Amount conventions.** SAP posts one signed GST-exclusive figure. Oracle posts unsigned amounts split across a debit column and a credit column, one of which is always null. The legacy system posts a single unsigned GST-inclusive figure, so the taxable value has to be derived by removing one eleventh before anything can be added up.
- **Entity keys.** `1010`, `LED_2001` and `AU01` are the same company. Nothing in the three extracts connects them; only `DIM_LEGAL_ENTITY` does, and it is also the only place that records which entities are absent from which system.
- **Account codes.** `400001`, `5100-0001` and `AP001` describe the same expense. Without the mapping there is no supply category, and without a supply category no tax treatment can be determined.

### One query, the whole diagnosis

Rather than run a separate count for each problem, ask for all of them at once: one row per defect, with a count and a percentage of the relevant population.

In [ ]:
-- One query, one row per defect, the whole diagnosis. Scans all 2,400,000
-- ledger lines plus the supplier and entity masters.
WITH gl AS (
    SELECT 'SAP' AS SOURCE_SYSTEM, HKONT AS NATIVE_ACCOUNT_CODE,
           LIFNR AS SUPPLIER_ID, MWSTS AS BOOKED_TAX
    FROM RAW_DATA.ERP_SAP_GL
    UNION ALL
    SELECT 'ORACLE', CODE_COMBINATION, VENDOR_ID, TAX_AMOUNT
    FROM RAW_DATA.ERP_ORACLE_GL
    UNION ALL
    SELECT 'LEGACY', acct, vendor_ref, tax_amt
    FROM RAW_DATA.ERP_LEGACY_AP
)
SELECT 'Legacy dates that will not parse'                    AS ISSUE,
       COUNT(*)                                              AS AFFECTED,
       ROUND(100.0 * COUNT(*)
             / (SELECT COUNT(*) FROM RAW_DATA.ERP_LEGACY_AP), 2) AS PCT_OF_POPULATION,
       'ERP_LEGACY_AP lines'                                 AS POPULATION
FROM RAW_DATA.ERP_LEGACY_AP
WHERE TRY_TO_DATE(txn_dt, 'DD/MM/YYYY') IS NULL

UNION ALL
SELECT 'GL lines with no account mapping', COUNT(*),
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gl), 2), 'All GL lines'
FROM gl g
LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
       ON m.SOURCE_SYSTEM = g.SOURCE_SYSTEM
      AND m.NATIVE_ACCOUNT_CODE = g.NATIVE_ACCOUNT_CODE
WHERE m.MAPPING_ID IS NULL

UNION ALL
SELECT 'GL lines with GST booked by an unregistered supplier', COUNT(*),
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gl), 2), 'All GL lines'
FROM gl g
JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = g.SUPPLIER_ID
WHERE s.GST_REGISTERED = FALSE AND ABS(g.BOOKED_TAX) > 0

UNION ALL
SELECT 'Suppliers with no ABN on file', COUNT(*),
       ROUND(100.0 * COUNT(*)
             / (SELECT COUNT(*) FROM RAW_DATA.DIM_SUPPLIER), 2), 'Supplier master records'
FROM RAW_DATA.DIM_SUPPLIER
WHERE SUPPLIER_ABN IS NULL

UNION ALL
SELECT 'Entities missing from at least one ERP', COUNT(*),
       ROUND(100.0 * COUNT(*)
             / (SELECT COUNT(*) FROM RAW_DATA.DIM_LEGAL_ENTITY), 2), 'Group legal entities'
FROM RAW_DATA.DIM_LEGAL_ENTITY
WHERE SAP_BUKRS IS NULL OR ORACLE_LEDGER_ID IS NULL OR LEGACY_ENTITY_CD IS NULL

ORDER BY AFFECTED DESC;

Expect exactly these figures. The data is generated from seeded random values over a fixed date window, so the numbers on your screen match the numbers here.

| Issue | Affected | Percentage | Population |
|---|---|---|---|
| GL lines with no account mapping | 77,484 | 3.23% | All GL lines |
| GL lines with GST booked by an unregistered supplier | 31,039 | 1.29% | All GL lines |
| Legacy dates that will not parse | 8,088 | 2.02% | `ERP_LEGACY_AP` lines |
| Suppliers with no ABN on file | 102 | 5.10% | Supplier master records |
| Entities missing from at least one ERP | 6 | 42.86% | Group legal entities |

**77,484 ledger lines carry an account code nobody has mapped.** Those lines have no supply category, so no rule can reach them, so their GST treatment is unknown. That is not a data hygiene footnote — it is an unquantified position in a lodged return.

*Snowflake: five defect measures over all 2,400,000 lines plus the supplier and entity masters, in one query, on the smallest warehouse available. No sampling and no extract.*

> 💡 **The work is not the analysis**
>
> The partner's question needed one `GROUP BY`. Everything that stands between you and a defensible answer to it is conformance: parsing dates, resolving entities, mapping accounts, normalising signs. That is what Step 3 builds, once, for every question that follows.

> **What the tax practice can do now**
>
> - [x] see all three ERP extracts side by side, unmodified — *Step 1*
> - [x] show exactly why the group GST position cannot be answered today — *Step 2*
> - [ ] query 2,400,000 lines as one governed ledger that maintains itself
> - [ ] read the entire determination logic without reading SQL
> - [ ] reconcile the computed position against what was actually lodged
> - [ ] reproduce any prior filing on demand, and prove nothing moved

---

## Step 3 of 8 — Conform three ERPs into one governed ledger

Inspect → Diagnose → **Conform** → Determine → Reconcile

*~14 min* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-3)

In Step 2 you proved that no single source system can answer the question. This step fixes that permanently. One object, every line from every system in one shape, with the dimensions attached and the data-quality problems labelled rather than hidden.

- Parse three different date formats without the query dying on the bad rows.
- Conform three amount conventions into one signed net amount.
- Resolve three entity identifier schemes to one group entity.
- Build one Dynamic Table that keeps itself up to date as the source systems change.

### Where strict parsing falls over

The legacy AP system stores its document date as text in `DD/MM/YYYY`. About 2% of those 400,000 values cannot be parsed at all: some are `'N/A'`, some are empty strings, and some are dates that look plausible but do not exist — `'31/02/2024'` among them. Run both statements below in order.

> ⚠️ **This cell is meant to fail**
>
> Run it anyway. The error is the teaching point, and the next cell fixes it.

In [ ]:
-- 1. The strict parser. This is EXPECTED TO FAIL - that is the lesson.
--    TO_DATE does not skip the bad rows and it does not return NULL for
--    them. One unparseable value out of 400,000 fails the entire query.
SELECT MIN(TO_DATE(txn_dt, 'DD/MM/YYYY')) AS EARLIEST_DOC_DATE
FROM RAW_DATA.ERP_LEGACY_AP;

In [ ]:
-- 2. The tolerant parser. Same column, same format mask, one word
--    different. Every row survives. The values that cannot be parsed
--    arrive as NULL, which means they can be counted, reported and
--    chased instead of taking the pipeline down with them.
SELECT txn_dt                            AS RAW_VALUE,
       TRY_TO_DATE(txn_dt, 'DD/MM/YYYY') AS TOLERANT_PARSE,
       COUNT(*)                          AS LINE_COUNT
FROM RAW_DATA.ERP_LEGACY_AP
WHERE TRY_TO_DATE(txn_dt, 'DD/MM/YYYY') IS NULL
GROUP BY 1, 2
ORDER BY 3 DESC;

Statement 1 stops with `Can't parse '' as date with format 'DD/MM/YYYY'`. One empty string, out of 400,000 rows, and there is no result at all — no partial answer, no list of the rows at fault. Statement 2 returns the three shapes that break it:

| Raw value | Tolerant parse | Lines | Why it fails |
|---|---|---|---|
| `N/A` | `NULL` | 2,799 | Not a date at all — an operator typed a placeholder. |
| `31/02/2024` | `NULL` | 2,871 | Correctly formatted and still impossible: February has no 31st. |
| *(empty string)* | `NULL` | 2,418 | The field was never populated. |
| **Total unparseable** |  | **8,088** | 2.02% of the legacy ledger. |

The middle case is the instructive one. `31/02/2024` passes any format check, any regular expression and any length validation you might write — it is only invalid because that day does not exist. A calendar-aware parser catches it; pattern matching never will.

> 💡 **The distinction that matters**
>
> `TO_DATE` asserts that the data is clean. `TRY_TO_DATE` asserts nothing and tells you where it is not.

### What has to be conformed, and why

The three systems disagree on more than dates. They disagree about how an amount is signed, what an amount includes, and how a legal entity is named. None of that can be left to the person writing the next query.

| Source | Document date | Amount convention | Entity identifier |
|---|---|---|---|
| **SAP** | `BUDAT` — text, `YYYYMMDD` | Signed amount in `DMBTR`, tax carried separately in `MWSTS` | `BUKRS` company code |
| **Oracle** | `GL_DATE` — native `DATE`, no parsing needed | Separate debit and credit columns, netted as `ENTERED_DR - ENTERED_CR` | `LEDGER_ID` |
| **Legacy AP** | `txn_dt` — text, `DD/MM/YYYY`, ~2% unparseable | GST-**inclusive** gross; net recovered as `amt_incl_tax - tax_amt` | `entity_cd` |

The amount conventions are the quietest of the three defects and the most dangerous. A signed amount, a debit-minus-credit pair and a GST-inclusive gross are all "the amount" to the system that produced them. Add them together untouched and the total is meaningless — and it will still look like a number, which is why nobody catches it. Each branch of the union below normalises to the same definition: `NET_AMOUNT` is always net of GST, and `BOOKED_TAX_AMOUNT` is always the tax the source system actually posted.

The entity crosswalk works the same way. `DIM_LEGAL_ENTITY` holds one row per group entity carrying all three identifiers — `SAP_BUKRS`, `ORACLE_LEDGER_ID` and `LEGACY_ENTITY_CD`. The `ent_xwalk` CTE unpivots that once into `(SOURCE_SYSTEM, SRC_ENTITY) → ENTITY_ID`, so the union can be joined a single time on a single pair of columns instead of carrying a three-branch `OR` condition. One group entity, three names, resolved in one place.

### Build the governed ledger

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE CURATED.GL_HARMONISED
    TARGET_LAG = '1 hour'
    WAREHOUSE  = GSI_TAX_WH
    REFRESH_MODE = AUTO
AS
WITH ent_xwalk AS (
    -- Unpivot the crosswalk once: three ERP identifiers, one group entity.
    SELECT ENTITY_ID, 'SAP'    AS SOURCE_SYSTEM, SAP_BUKRS        AS SRC_ENTITY
      FROM RAW_DATA.DIM_LEGAL_ENTITY WHERE SAP_BUKRS        IS NOT NULL
    UNION ALL
    SELECT ENTITY_ID, 'ORACLE', ORACLE_LEDGER_ID FROM RAW_DATA.DIM_LEGAL_ENTITY
     WHERE ORACLE_LEDGER_ID IS NOT NULL
    UNION ALL
    SELECT ENTITY_ID, 'LEGACY', LEGACY_ENTITY_CD FROM RAW_DATA.DIM_LEGAL_ENTITY
     WHERE LEGACY_ENTITY_CD IS NOT NULL
),
u AS (
    -- SAP: signed amounts, YYYYMMDD text dates
    SELECT 'SAP' AS SOURCE_SYSTEM, BELNR AS SOURCE_DOC_REF, BUZEI AS SOURCE_LINE_NUM,
           BUKRS AS SRC_ENTITY, HKONT AS NATIVE_ACCOUNT_CODE,
           TRY_TO_DATE(BUDAT, 'YYYYMMDD') AS DOC_DATE, DOC_TYPE,
           DMBTR AS NET_AMOUNT, MWSTS AS BOOKED_TAX_AMOUNT,
           WAERS AS CURRENCY, MWSKZ AS BOOKED_TAX_CODE,
           LIFNR AS SUPPLIER_ID, SGTXT AS LINE_NARRATIVE
    FROM RAW_DATA.ERP_SAP_GL
    UNION ALL
    -- Oracle: native DATE, separate DR/CR columns
    SELECT 'ORACLE', JE_HEADER_ID::VARCHAR, JE_LINE_NUM,
           LEDGER_ID, CODE_COMBINATION,
           GL_DATE, DOC_TYPE,
           COALESCE(ENTERED_DR, 0) - COALESCE(ENTERED_CR, 0), TAX_AMOUNT,
           CURRENCY_CODE, TAX_CLASSIFICATION,
           VENDOR_ID, DESCRIPTION
    FROM RAW_DATA.ERP_ORACLE_GL
    UNION ALL
    -- Legacy: DD/MM/YYYY text dates, GST-INCLUSIVE amounts (gross up removed)
    SELECT 'LEGACY', doc_ref, 1,
           entity_cd, acct,
           TRY_TO_DATE(txn_dt, 'DD/MM/YYYY'), doc_type,
           amt_incl_tax - tax_amt, tax_amt,
           curr_cd, tax_flag,
           vendor_ref, narrative
    FROM RAW_DATA.ERP_LEGACY_AP
)
SELECT
    u.SOURCE_SYSTEM || '-' || u.SOURCE_DOC_REF || '-' || u.SOURCE_LINE_NUM::VARCHAR AS GL_LINE_ID,
    u.SOURCE_SYSTEM, u.SOURCE_DOC_REF, u.SOURCE_LINE_NUM,
    x.ENTITY_ID, e.ENTITY_NAME, e.JURISDICTION, e.ABN AS ENTITY_ABN,
    u.DOC_DATE,
    DATE_TRUNC('month', u.DOC_DATE)                              AS TAX_PERIOD,
    u.DOC_TYPE,
    u.NATIVE_ACCOUNT_CODE,
    m.GROUP_ACCOUNT_CODE, m.GROUP_ACCOUNT_NAME, m.SUPPLY_CATEGORY,
    u.SUPPLIER_ID, s.SUPPLIER_NAME, s.SUPPLIER_ABN,
    s.GST_REGISTERED                                             AS SUPPLIER_GST_REGISTERED,
    s.SUPPLIER_COUNTRY,
    (s.SUPPLIER_COUNTRY <> e.JURISDICTION)                       AS IS_CROSS_BORDER,
    u.CURRENCY,
    u.NET_AMOUNT,
    u.BOOKED_TAX_AMOUNT,
    u.NET_AMOUNT + u.BOOKED_TAX_AMOUNT                           AS GROSS_AMOUNT,
    u.BOOKED_TAX_CODE,
    u.LINE_NARRATIVE,
    -- conformance flags: is the DATA complete? (not: is the TAX right?)
    (u.DOC_DATE          IS NOT NULL)                            AS IS_DATE_PARSED,
    (x.ENTITY_ID         IS NOT NULL)                            AS IS_ENTITY_RESOLVED,
    (m.SUPPLY_CATEGORY   IS NOT NULL)                            AS IS_ACCOUNT_MAPPED,
    (s.SUPPLIER_ID       IS NOT NULL)                            AS IS_SUPPLIER_RESOLVED
FROM u
LEFT JOIN ent_xwalk x
       ON x.SOURCE_SYSTEM = u.SOURCE_SYSTEM AND x.SRC_ENTITY = u.SRC_ENTITY
LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e   ON e.ENTITY_ID = x.ENTITY_ID
LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
       ON m.SOURCE_SYSTEM = u.SOURCE_SYSTEM AND m.NATIVE_ACCOUNT_CODE = u.NATIVE_ACCOUNT_CODE
LEFT JOIN RAW_DATA.DIM_SUPPLIER s       ON s.SUPPLIER_ID = u.SUPPLIER_ID;

Notice the shape: three `SELECT` branches unioned inside a CTE, then the dimension joins applied **once** over the union rather than repeated per branch. Add a fourth source system and you add one branch and one crosswalk row. Nothing else in the statement changes.

Notice also that every dimension join is a `LEFT JOIN`. An inner join here would quietly delete the lines with an unmapped account code, and those are precisely the lines the tax team needs to see. The four conformance flags at the bottom exist so that incompleteness is recorded as a fact about the row rather than resolved by dropping it.

> 💡 **The account mapping is a JOIN, not a CASE statement**
>
> `SUPPLY_CATEGORY` arrives from `DIM_ACCOUNT_MAPPING` through a join on `(SOURCE_SYSTEM, NATIVE_ACCOUNT_CODE)`. It would have been shorter to write a `CASE` expression listing account codes inline, and it would have been wrong.

One more thing this table deliberately does not contain: any tax logic. There are no rates, no treatments and no judgements about whether GST was handled correctly. That is because the same conformed ledger feeds statutory reporting, management reporting and tax, and the moment a tax assumption is baked into it, the other two consumers inherit it silently.

> ✅ **What one CREATE statement just bought you**
>
> That single statement produced a working pipeline: it unions three source systems, resolves four dimensions and labels four data-quality conditions across **2,400,000 rows** — and it keeps itself up to date as the sources change. There is no scheduler to configure, no orchestration tool to install, no dependency graph to maintain by hand and nothing to deploy.

Three things worth doing on screen before you move on, none of which needs a query. First, open **Monitoring → Dynamic Tables** in Snowsight: the refresh history and the dependency graph are already there, because the pipeline draws its own diagram. Second, when you run the verification query below, open the **Query Profile** tab on the result — a visual execution plan with partition pruning statistics, for free, on a query you were running anyway. Third, run that verification query a second time: it returns instantly from the result cache and consumes no compute at all.

### One query that verifies the whole build

In [ ]:
SELECT COUNT(*)                                                    AS TOTAL_ROWS,
       COUNT(DISTINCT GL_LINE_ID)                                  AS DISTINCT_LINES,
       ROUND(100.0 * SUM(IFF(IS_DATE_PARSED, 1, 0))      / COUNT(*), 2) AS PCT_DATE_OK,
       ROUND(100.0 * SUM(IFF(IS_ENTITY_RESOLVED, 1, 0))  / COUNT(*), 2) AS PCT_ENTITY_OK,
       ROUND(100.0 * SUM(IFF(IS_ACCOUNT_MAPPED, 1, 0))   / COUNT(*), 2) AS PCT_ACCOUNT_OK,
       ROUND(100.0 * SUM(IFF(IS_SUPPLIER_RESOLVED, 1, 0))/ COUNT(*), 2) AS PCT_SUPPLIER_OK
FROM CURATED.GL_HARMONISED;

| Measure | Result | What it tells you |
|---|---|---|
| `TOTAL_ROWS` | **2,400,000** | 1,200,000 SAP + 800,000 Oracle + 400,000 legacy. Nothing lost. |
| `DISTINCT_LINES` | **2,400,000** | Identical to the row count, so no dimension join has duplicated a line. |
| `PCT_DATE_OK` | **99.66%** | The 0.34% shortfall is the unparseable legacy dates, now flagged rather than lost. |
| `PCT_ENTITY_OK` | **100.00%** | Every line resolved to a group entity through the crosswalk. |
| `PCT_ACCOUNT_OK` | **96.77%** | 3.23% of the ledger carries an account code nobody has mapped. |
| `PCT_SUPPLIER_OK` | **100.00%** | Every line resolved to a supplier record. |

> ⚠️ **The fan-out check is not a formality**
>
> `COUNT(*) = COUNT(DISTINCT GL_LINE_ID)` is the assertion that no join multiplied a ledger line. If a dimension table ever gains a duplicate key, this is the number that moves — and it moves before anyone notices that the GST total has grown.

### Inspect the object you created

In [ ]:
SHOW DYNAMIC TABLES IN SCHEMA CURATED;

Read across the row: the target lag you declared, the warehouse that maintains it, the scheduling state and the refresh mode Snowflake actually chose. You asked for `REFRESH_MODE = AUTO`, which means the platform decides how to keep the table current rather than you deciding for it — so read what it reports rather than assuming a value. `TARGET_LAG` is a freshness objective, not a schedule: it is a promise about how stale the data may become, not an instruction to run at a fixed time.

> ✅ **One ledger, and no tax logic in it**
>
> One governed ledger, 2,400,000 lines, three source systems, four dimensions resolved and four data-quality conditions labelled. It contains no tax logic whatsoever — and that is what makes the next step possible.

---

## Step 4 of 8 — The tax rules layer

Inspect → Diagnose → Conform → **Determine** → Reconcile

*~15 min* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-4)

**The rules belong to the tax team, not to a developer.** Tax logic is not a `CASE` statement. It is a body of rules with jurisdictions, effective dates and precedence, and it changes at short notice, usually decided by someone well outside IT. So hold it as data the tax team can read, review and change themselves.

By the end of this step you will:

- Read the entire determination logic of the system without reading any SQL.
- Apply effective-dated rules with generic SQL that contains no rates and no treatments.
- Move the group GST position with a single `UPDATE` to one row of data.
- Stand up a self-maintaining exception register over the result.

### The rules, as data

In [ ]:
-- The decision table. NULL means "matches anything". Lower PRIORITY wins.
SELECT RULE_ID, PRIORITY, JURISDICTION, SUPPLY_CATEGORY,
       SUPPLIER_GST_REGISTERED, IS_CROSS_BORDER,
       TAX_TREATMENT, RECOVERABLE,
       VALID_FROM, VALID_TO, RULE_DESCRIPTION
FROM TAX.TAX_RULE_DETERMINATION
ORDER BY PRIORITY, RULE_ID;

In [ ]:
-- The rate table. Keyed on jurisdiction, treatment and date.
SELECT RATE_ID, JURISDICTION, TAX_TREATMENT, RATE_PCT, VALID_FROM, VALID_TO
FROM TAX.TAX_RULE_RATE
ORDER BY JURISDICTION, VALID_FROM, TAX_TREATMENT;

**A tax person can read those two grids and understand the whole determination logic without reading a line of SQL.** Sixteen rules, nineteen rates. No code review, no developer to interpret, nothing buried in a query somebody wrote four years ago and left.

Find **rules 9 and 16**. Same priority, same predicates — `PROFESSIONAL_FEES`, supplier not GST registered, cross-border — and two adjacent, non-overlapping validity windows:

| Rule | Priority | Predicates | Treatment | Recoverable | Valid from | Valid to |
|---|---|---|---|---|---|---|
| **9** | 20 | `PROFESSIONAL_FEES`, unregistered, cross-border | **GST_FREE** | FALSE | 2000-07-01 | **2024-06-30** |
| **16** | 20 | `PROFESSIONAL_FEES`, unregistered, cross-border | **REVERSE_CHARGE** | TRUE | **2024-07-01** | 9999-12-31 |

That is not two rules. It is **the same rule with two effective-dated versions**. Following a review, the group corrected its position on imported professional services with effect from 1 July 2024 — before that date those acquisitions were treated as out of scope, and from that date the reverse charge applies. Unrecognised reverse charge on imported services is one of the most common findings in an Australian GST review, so this is not a hypothetical.

> ⚠️ **Worth pausing on before you run anything else**
>
> What happens to three years of already-lodged BAS if that corrected treatment is *hard-coded* instead of effective-dated? You do not merely get next quarter wrong. Your computed position for closed periods moves, your reconciliation lights up for periods that were correct when they were filed, and you can no longer distinguish a genuine variance from one you manufactured yourself.

Two details on the rate table while it is on screen. The Australian GST rate has been 10% since 1 July 2000 and has never changed, so what moves in this workshop is a *treatment*, not a rate — because treatment is what actually moves in Australia. The table is date-effective regardless, so if the rate ever does change, nobody edits a query.

And rule 15 matches everything at priority 99. That catch-all is what guarantees every line resolves to exactly one rule. A decision table without one silently drops rows, which is the worst failure mode in a tax computation because the total still looks reasonable.

### Why two rule tables and not one

Different grain. Determination is keyed on the characteristics of the transaction — jurisdiction, supply category, whether the supplier is registered, whether the supply is cross-border. Rate is keyed on jurisdiction, treatment and date. Collapse them into one table and a single rate change becomes an update across every rule that happens to mention that treatment, which is exactly the kind of multi-row edit that gets one row wrong.

> 💡 **And why the rules are not SQL strings in a VARCHAR**
>
> A recurring suggestion is to store each rule as a SQL fragment in a text column and evaluate it dynamically. Do not: a rule expressed as a string cannot be tested, validated on entry, or type-checked, so the first time you learn it is malformed is when it runs in a production computation. It is also a straightforward injection surface, because anyone who can edit a rule row can then execute arbitrary SQL with the pipeline's privileges. Rules here are structured columns with declared types, which is why they can be constrained, reviewed and diffed.

### Build the determination view

In [ ]:
CREATE OR REPLACE VIEW TAX.V_GST_DETERMINED AS
WITH matched AS (
    SELECT
        g.*,
        r.RULE_ID          AS APPLIED_RULE_ID,
        r.RULE_DESCRIPTION AS APPLIED_RULE_DESCRIPTION,
        r.TAX_TREATMENT,
        r.RECOVERABLE
    FROM CURATED.GL_HARMONISED g
    LEFT JOIN TAX.TAX_RULE_DETERMINATION r
      ON (r.JURISDICTION            IS NULL OR r.JURISDICTION            = g.JURISDICTION)
     AND (r.SUPPLY_CATEGORY         IS NULL OR r.SUPPLY_CATEGORY         = g.SUPPLY_CATEGORY)
     AND (r.SUPPLIER_GST_REGISTERED IS NULL OR r.SUPPLIER_GST_REGISTERED = g.SUPPLIER_GST_REGISTERED)
     AND (r.IS_CROSS_BORDER         IS NULL OR r.IS_CROSS_BORDER         = g.IS_CROSS_BORDER)
     AND  g.DOC_DATE BETWEEN r.VALID_FROM AND r.VALID_TO
    QUALIFY ROW_NUMBER() OVER (PARTITION BY g.GL_LINE_ID
                               ORDER BY r.PRIORITY, r.RULE_ID) = 1
)
SELECT
    m.*,
    rt.RATE_PCT                                          AS APPLIED_RATE_PCT,
    ROUND(m.NET_AMOUNT * rt.RATE_PCT / 100, 2)           AS EXPECTED_TAX_AMOUNT,
    ROUND(m.BOOKED_TAX_AMOUNT
          - ROUND(m.NET_AMOUNT * rt.RATE_PCT / 100, 2), 2) AS TAX_VARIANCE
FROM matched m
LEFT JOIN TAX.TAX_RULE_RATE rt
       ON rt.JURISDICTION   = m.JURISDICTION
      AND rt.TAX_TREATMENT  = m.TAX_TREATMENT
      AND m.DOC_DATE BETWEEN rt.VALID_FROM AND rt.VALID_TO;

Read what is *not* in that statement. There are no rates, no jurisdictions, no supply categories and no treatments anywhere in it. It is a thin, generic mechanism for applying whatever the two tables happen to say. Change a rule and you have changed the answer this view returns on the next query — no rebuild, no redeployment.

Both joins are date-ranged on `DOC_DATE` and not on `CURRENT_DATE()`. This is the whole idea: a transaction is judged by the rules that applied *when it happened*. It is also why rules 9 and 16 can share priority 20 without ambiguity — their windows do not overlap, so only one of them can ever match a given line.

> 💡 **QUALIFY, in one paragraph**
>
> If you have not met `QUALIFY` before: It filters on the result of a window function, in the same way `HAVING` filters on the result of an aggregate. Here, a single ledger line can legitimately match several rules — an entertainment line matches rule 1 and also the rule 15 catch-all. `ROW_NUMBER() OVER (PARTITION BY g.GL_LINE_ID ORDER BY r.PRIORITY, r.RULE_ID)` numbers each line's candidate rules from most specific to least, and `QUALIFY ... = 1` keeps only the winner. That is most-specific-wins precedence expressed in two lines, with no self-join and no subquery.

The join to the rule table is a `LEFT JOIN`, deliberately. A line whose date could not be parsed has no determinable treatment — you genuinely cannot say how a transaction should be taxed when you do not know when it occurred. With an inner join those lines vanish silently, which is catastrophic, because they are precisely the lines somebody needs to chase. As a left join all 2,400,000 lines survive and `TAX_TREATMENT` comes back `NULL` only where `IS_DATE_PARSED = FALSE`. Missing information is represented as missing information.

### Prove the effective dating

One query, one population: Australian imported professional fees from offshore suppliers that are not GST registered. Grouped either side of 1 July 2024, from the same view, with no conditional tax logic anywhere.

In [ ]:
SELECT CASE WHEN DOC_DATE < '2024-07-01'::DATE
            THEN 'Before 1 Jul 2024'
            ELSE 'From 1 Jul 2024' END      AS PERIOD_SIDE,
       APPLIED_RULE_ID,
       TAX_TREATMENT,
       APPLIED_RATE_PCT,
       COUNT(*)                             AS LINE_COUNT,
       ROUND(SUM(EXPECTED_TAX_AMOUNT), 2)   AS EXPECTED_GST
FROM TAX.V_GST_DETERMINED
WHERE JURISDICTION            = 'AU'
  AND SUPPLY_CATEGORY         = 'PROFESSIONAL_FEES'
  AND SUPPLIER_GST_REGISTERED = FALSE
  AND IS_CROSS_BORDER         = TRUE
  AND DOC_DATE IS NOT NULL
GROUP BY 1, 2, 3, 4
ORDER BY 1;

| Period | Rule | Treatment | Rate | Lines | Expected GST |
|---|---|---|---|---|---|
| Before 1 Jul 2024 | **9** | `GST_FREE` | 0.00% | 2,429 | **$0** |
| From 1 Jul 2024 | **16** | `REVERSE_CHARGE` | 10.00% | 4,690 | **$26,247,261** |

An otherwise identical population, split only by document date, produces two different treatments and **$26.2 million** of expected GST on one side of the line that does not exist on the other. Nothing in the query decided that. The rows in the rule table decided it.

> ⚠️ **Now make the consequence explicit**
>
> Had the corrected treatment been hard-coded — a `CASE` expression saying imported professional fees are reverse charged — the *before* row above would read `REVERSE_CHARGE` at 10.00% as well. Every period already lodged on the old position would be silently restated, and the reconciliation in Step 5 would raise variances against returns that were correct when they were filed. Effective dating is not bookkeeping neatness. It is the only thing standing between you and restating three years of lodged BAS by accident.

### Change one row and watch the position move

So what happens when the tax team wants a change? Do it and see. Run all three statements in order.

In [ ]:
-- 1. The tax team wants to model the reverse charge at a different rate.
--    They change one row of data. Not a query. Not a deployment.
UPDATE TAX.TAX_RULE_RATE
   SET RATE_PCT = 15.00
 WHERE JURISDICTION = 'AU' AND TAX_TREATMENT = 'REVERSE_CHARGE';

In [ ]:
-- 2. Exactly the same aggregate you just ran. Nothing was rebuilt,
--    nothing was refreshed, nothing was redeployed.
SELECT APPLIED_RULE_ID,
       TAX_TREATMENT,
       APPLIED_RATE_PCT,
       COUNT(*)                             AS LINE_COUNT,
       ROUND(SUM(EXPECTED_TAX_AMOUNT), 2)   AS EXPECTED_GST
FROM TAX.V_GST_DETERMINED
WHERE JURISDICTION            = 'AU'
  AND SUPPLY_CATEGORY         = 'PROFESSIONAL_FEES'
  AND SUPPLIER_GST_REGISTERED = FALSE
  AND IS_CROSS_BORDER         = TRUE
  AND DOC_DATE >= '2024-07-01'::DATE
GROUP BY 1, 2, 3;

In [ ]:
-- 3. Put it back. The Australian rate is 10% and has been since 2000.
UPDATE TAX.TAX_RULE_RATE
   SET RATE_PCT = 10.00
 WHERE JURISDICTION = 'AU' AND TAX_TREATMENT = 'REVERSE_CHARGE';

**The group's computed GST position moved twice in about four seconds.** `APPLIED_RATE_PCT` reads 15.00 and expected GST on those 4,690 lines rises by half to $39,370,891. Statement 3 puts it back and the position returns to $26,247,261.

*Snowflake: the view is generic, so a rule change takes effect on the next query. No rebuild, no refresh, no redeployment.*

> ✅ **This is ownership, not architecture**
>
> No code was edited. Nothing was rebuilt, nothing was redeployed, no release was cut and no developer was involved. **The person who owns the rule changed the rule.** That is the whole point of holding tax logic as data: the people accountable for the position can move the position, and everyone else can see exactly what they changed and when it takes effect.

### The standing exception register

One view that turns the determined ledger into a work queue: every line that fails a tax test, with the reasons concatenated and a risk score to sort by.

In [ ]:
CREATE OR REPLACE VIEW TAX.V_GST_EXCEPTIONS AS
WITH d AS (SELECT * FROM TAX.V_GST_DETERMINED),
ex AS (
    SELECT GL_LINE_ID, 'E1' AS EXCEPTION_CODE, 'MEDIUM' AS SEVERITY,
           'Account code ' || COALESCE(NATIVE_ACCOUNT_CODE,'?') || ' has no group mapping' AS EXCEPTION_REASON
    FROM d WHERE NOT IS_ACCOUNT_MAPPED
    UNION ALL
    SELECT GL_LINE_ID, 'E2', 'HIGH', 'Document date could not be parsed from the source system'
    FROM d WHERE NOT IS_DATE_PARSED   -- reachable only because of the LEFT JOIN in V_GST_DETERMINED
    UNION ALL
    SELECT GL_LINE_ID, 'E3', 'HIGH', 'Supplier ABN missing on a claimed credit over $82.50'
    FROM d WHERE SUPPLIER_ABN IS NULL AND DOC_TYPE = 'AP_INVOICE'
             AND RECOVERABLE AND ABS(GROSS_AMOUNT) > 82.50
    UNION ALL
    SELECT GL_LINE_ID, 'E4', 'HIGH', 'GST charged by a supplier that is not GST registered'
    FROM d WHERE SUPPLIER_GST_REGISTERED = FALSE AND BOOKED_TAX_AMOUNT <> 0
    UNION ALL
    SELECT GL_LINE_ID, 'E5', 'MEDIUM',
           'Booked GST differs from expected by ' || TO_VARCHAR(ROUND(TAX_VARIANCE,2))
    FROM d WHERE ABS(COALESCE(TAX_VARIANCE, 0)) > 1.00
    UNION ALL
    SELECT GL_LINE_ID, 'E6', 'HIGH',
           'Credit claimed on a non-recoverable treatment (' || TAX_TREATMENT || ')'
    FROM d WHERE RECOVERABLE = FALSE AND TAX_TREATMENT <> 'NON_DEDUCTIBLE'
             AND DOC_TYPE = 'AP_INVOICE' AND BOOKED_TAX_AMOUNT <> 0
    UNION ALL
    SELECT GL_LINE_ID, 'E7', 'MEDIUM', 'Reverse charge required but not applied in the source system'
    FROM d WHERE TAX_TREATMENT = 'REVERSE_CHARGE'
             AND COALESCE(BOOKED_TAX_CODE,'') NOT IN ('RC','REVERSE_CHARGE')
)
SELECT
    d.GL_LINE_ID, d.ENTITY_ID, d.ENTITY_NAME, d.TAX_PERIOD,
    d.SUPPLIER_ID, d.SUPPLIER_NAME,
    d.SOURCE_SYSTEM, d.SOURCE_DOC_REF, d.DOC_TYPE, d.SUPPLY_CATEGORY,
    d.NET_AMOUNT, d.BOOKED_TAX_AMOUNT, d.EXPECTED_TAX_AMOUNT, d.TAX_VARIANCE,
    d.TAX_TREATMENT, d.APPLIED_RULE_ID,
    COUNT(*)                                                        AS EXCEPTION_COUNT,
    LISTAGG(DISTINCT ex.EXCEPTION_CODE, ', ')
        WITHIN GROUP (ORDER BY ex.EXCEPTION_CODE)                   AS EXCEPTION_CODES,
    LISTAGG(ex.EXCEPTION_REASON, ' | ')
        WITHIN GROUP (ORDER BY ex.EXCEPTION_CODE)                   AS EXCEPTION_REASONS,
    CASE MAX(CASE ex.SEVERITY WHEN 'HIGH' THEN 3 WHEN 'MEDIUM' THEN 2 ELSE 1 END)
         WHEN 3 THEN 'HIGH' WHEN 2 THEN 'MEDIUM' ELSE 'LOW' END     AS MAX_SEVERITY,
    ROUND(COUNT(*) * 10
          + LEAST(ABS(COALESCE(d.TAX_VARIANCE, 0)) / 100, 50), 2)   AS RISK_SCORE
FROM ex
JOIN d ON d.GL_LINE_ID = ex.GL_LINE_ID
GROUP BY ALL;

In [ ]:
-- The work queue: worst first.
SELECT GL_LINE_ID, ENTITY_NAME, TAX_PERIOD, SUPPLIER_NAME,
       SUPPLY_CATEGORY, TAX_TREATMENT, APPLIED_RULE_ID,
       NET_AMOUNT, BOOKED_TAX_AMOUNT, EXPECTED_TAX_AMOUNT, TAX_VARIANCE,
       EXCEPTION_COUNT, EXCEPTION_CODES, MAX_SEVERITY, RISK_SCORE
FROM TAX.V_GST_EXCEPTIONS
ORDER BY RISK_SCORE DESC
LIMIT 50;

That is a standing exception register: seven tax tests over the whole ledger, always current, always sorted worst-first.

| Code | Test | Lines |
|---|---|---|
| `E1` | Account code has no group mapping | 77,484 |
| `E2` | Document date could not be parsed | 8,088 |
| `E3` | Supplier ABN missing on a claimed credit over $82.50 | 60,097 |
| `E4` | GST charged by a supplier that is not registered | 31,039 |
| `E5` | Booked GST differs from expected by more than $1 | 107,808 |
| `E6` | Credit claimed on a non-recoverable treatment | 36,718 |
| `E7` | Reverse charge required but not applied at source | 11,754 |
| **Distinct ledger lines flagged** |  | **253,394 — 10.56% of the ledger** |

A line can fail more than one test, which is why the code counts sum to more than 253,394 and why `EXCEPTION_COUNT` feeds the risk score. Note that `E2` returns exactly the 8,088 unparseable dates from Step 3 — the left join in the determination view is what allowed those lines to reach this register at all. Lab 2 comes back to this queue and uses Cortex to explain each flagged line in plain English.

> ✅ **What you just built**
>
> Sixteen determination rules and nineteen rates, held as data the tax team owns and can change without a developer; a thin generic view that applies whichever rules were in force on the date of each transaction; and a standing exception register over 2,400,000 ledger lines.

### This is one rule set, not *the* rule set

What you built is not "the tax rules". It is **one rule set, for one tax**, sitting beside a ledger that knows nothing about GST — which is exactly why tax logic was kept out of the curated layer in Step 3.

Every other indirect tax your practice touches has the same shape: rules the business owns, effective-dated, applied by thin generic SQL. Fuel tax credits reindex twice a year. Payroll tax is eight jurisdictions, each with its own rate, threshold and grouping rules. Withholding tax is a treaty rate by country and income type. Same pattern, different table.

**The ledger is bespoke to the client. The rule sets are not.** The payroll tax rule set you build for one client is most of the payroll tax rule set for the next one.

> ✅ **The commercial point**
>
> Most of this work today is billed by the hour and thrown away when the engagement closes. Build it this way and the rule libraries become your firm's own intellectual property — something you deploy repeatedly rather than rebuild from scratch for every client.

> ⚠️ **The honest boundary**
>
> Not everything is a rule table. Anything requiring iterative or circular computation needs procedural logic — thin capitalisation, Pillar Two top-up calculations, and consolidation eliminations with circular ownership are all in that category. Say so plainly. Acknowledging where the pattern stops is what makes the claim credible everywhere else.

---

## Step 5 of 8 — Reconcile, then survive the audit

Inspect → Diagnose → Conform → Determine → **Reconcile**

*~15 min* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-5)

You now have a computed GST position for every entity and every period. The client has a lodged one, sitting in `RAW_DATA.FCT_BAS_LODGED` — 532 rows, fourteen entities across thirty-eight complete months. Nobody has ever put the two side by side, because until this morning there was no computed position to compare against.

Then, halfway through the reconciliation, someone posts a backdated adjustment into the ledger — and the auditor asks the question every tax practitioner dreads: *show me the ledger exactly as it stood when we lodged.*

- Reconcile computed GST against lodged BAS at entity and period, and classify the result.
- Watch a backdated posting break a reconciliation that tied five minutes ago.
- Recover the as-lodged position with one clause of SQL — no backup, no restore.
- Take an immutable audit snapshot of a 2,400,000-row ledger, instantly and for no storage.
- Undo a `DROP TABLE` you should not have run.

### Build the reconciliation and run it

The view below is deliberately conservative in three places. It restricts to `JURISDICTION = 'AU'`, because the lodged data is Australian BAS. It excludes the current, incomplete month, which has no lodged BAS yet and would otherwise appear as a large false variance every time the workshop is run. And it uses a `FULL OUTER JOIN`, so a period that exists on one side only still appears rather than vanishing — a lodged period with no computed lines is exactly as interesting as the reverse.

It also tiers into three bands, not two: variance within $100 is `MATCHED`, within $1,000 is `REVIEW`, and anything larger is `INVESTIGATE`. Nothing lands in `REVIEW` with this data, and that is worth noticing — the variances here are either exactly nil or run to hundreds of thousands. Real client ledgers are rarely so tidy, which is why the middle band is there.

In [ ]:
CREATE OR REPLACE VIEW TAX.V_GST_VS_BAS AS
WITH computed AS (
    SELECT ENTITY_ID, ENTITY_NAME, TAX_PERIOD,
           ABS(COALESCE(SUM(CASE WHEN DOC_TYPE='AR_INVOICE'
                                 THEN EXPECTED_TAX_AMOUNT END), 0)) AS COMPUTED_1A,
           ABS(COALESCE(SUM(CASE WHEN DOC_TYPE='AP_INVOICE' AND RECOVERABLE
                                 THEN EXPECTED_TAX_AMOUNT END), 0)) AS COMPUTED_1B
    FROM TAX.V_GST_DETERMINED
    WHERE JURISDICTION = 'AU' AND TAX_PERIOD IS NOT NULL
      -- exclude the current, incomplete month: it has no lodged BAS yet and
      -- would otherwise show as a false variance
      AND TAX_PERIOD < DATE_TRUNC('month', '2026-07-31'::DATE)
    GROUP BY ENTITY_ID, ENTITY_NAME, TAX_PERIOD
)
SELECT
    COALESCE(c.ENTITY_ID, b.ENTITY_ID)                              AS ENTITY_ID,
    COALESCE(c.ENTITY_NAME, e.ENTITY_NAME)                          AS ENTITY_NAME,
    COALESCE(c.TAX_PERIOD, b.TAX_PERIOD)                            AS TAX_PERIOD,
    ROUND(COALESCE(c.COMPUTED_1A, 0), 2)                            AS COMPUTED_1A,
    ROUND(COALESCE(c.COMPUTED_1B, 0), 2)                            AS COMPUTED_1B,
    ROUND(COALESCE(c.COMPUTED_1A, 0) - COALESCE(c.COMPUTED_1B, 0), 2) AS COMPUTED_NET_GST,
    COALESCE(b.LABEL_1A_GST_ON_SALES, 0)                            AS LODGED_1A,
    COALESCE(b.LABEL_1B_GST_CREDITS, 0)                             AS LODGED_1B,
    COALESCE(b.NET_GST, 0)                                          AS LODGED_NET_GST,
    ROUND(COALESCE(c.COMPUTED_1A,0) - COALESCE(b.LABEL_1A_GST_ON_SALES,0), 2) AS VARIANCE_1A,
    ROUND(COALESCE(c.COMPUTED_1B,0) - COALESCE(b.LABEL_1B_GST_CREDITS,0), 2)  AS VARIANCE_1B,
    ROUND((COALESCE(c.COMPUTED_1A,0) - COALESCE(c.COMPUTED_1B,0))
          - COALESCE(b.NET_GST,0), 2)                               AS VARIANCE_NET,
    ROUND(((COALESCE(c.COMPUTED_1A,0) - COALESCE(c.COMPUTED_1B,0)) - COALESCE(b.NET_GST,0))
          / NULLIF(ABS(COALESCE(b.NET_GST,0)), 0) * 100, 2)         AS VARIANCE_PCT,
    CASE
      WHEN ABS((COALESCE(c.COMPUTED_1A,0)-COALESCE(c.COMPUTED_1B,0))
               - COALESCE(b.NET_GST,0)) <= 100  THEN 'MATCHED'
      WHEN ABS((COALESCE(c.COMPUTED_1A,0)-COALESCE(c.COMPUTED_1B,0))
               - COALESCE(b.NET_GST,0)) <= 1000 THEN 'REVIEW'
      ELSE 'INVESTIGATE'
    END                                                             AS RECON_STATUS
FROM computed c
FULL OUTER JOIN RAW_DATA.FCT_BAS_LODGED b
     ON b.ENTITY_ID = c.ENTITY_ID AND b.TAX_PERIOD = c.TAX_PERIOD
LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e
     ON e.ENTITY_ID = COALESCE(c.ENTITY_ID, b.ENTITY_ID);

In [ ]:
-- The reconciliation. One row per entity-period, worst first.
SELECT RECON_STATUS,
       COUNT(*)                        AS ENTITY_PERIODS,
       ROUND(SUM(ABS(VARIANCE_NET)),2) AS TOTAL_ABS_VARIANCE
FROM TAX.V_GST_VS_BAS
GROUP BY RECON_STATUS
ORDER BY TOTAL_ABS_VARIANCE DESC;

| Recon status | Entity-periods | What it means |
|---|---|---|
| `MATCHED` | **500** | Variance of exactly **$0.00**. Not "within tolerance" — nil. |
| `INVESTIGATE` | **32** | Variance above $1,000. These are the conversation. |
| **532 entity-periods** |  | Fourteen entities across thirty-eight complete months. |

**500 of 532 entity-periods agree to the cent.** Two independently produced numbers — one computed from 2,400,000 ledger lines through a rule set the tax team owns, one lodged with the ATO months or years ago — and the variance is exactly nil, not "within tolerance". That agreement is the evidence the pipeline is right, and it is what makes the remaining 32 worth taking seriously. A reconciliation that flags everything tells you nothing; this one flags 6% of periods and defends the other 94%.

> 💡 **What the earlier steps were for**
>
> Every step so far was investment: conforming three ERPs, resolving entities, externalising the rules. This single query is the return. It answers *"is our lodged position defensible?"* across the whole group, at line-level granularity, in seconds — and it will answer it again tomorrow without anyone touching it. Compare that against how long the same reconciliation takes today, and for how many entities.

### The adjustment lands

Now the realistic complication. Somebody in the business posts a backdated journal against a period that was lodged six months ago. They are not doing anything wrong: late invoices, accruals reversed, an intercompany correction. It happens constantly, and the tax team almost never hears about it.

In [ ]:
-- 1. Mark the moment. Everything before this timestamp is the ledger as it
--    stood when the BAS was lodged. This is what makes the next step work:
--    you are not guessing how long ago something happened, you named it.
SET pre_adjustment = (SELECT CURRENT_TIMESTAMP());

In [ ]:
-- 2. A backdated AR adjustment against ENT001 (SAP company code 1010),
--    dated into the month that closed six months ago. AR invoices are
--    booked negative in SAP, so both amounts carry a minus sign.
INSERT INTO RAW_DATA.ERP_SAP_GL
    (BELNR, BUZEI, BUKRS, HKONT, BUDAT, DMBTR, MWSTS, WAERS, MWSKZ, LIFNR, DOC_TYPE, SGTXT)
SELECT 'SAP9900001', 1, '1010', '400001',
       TO_VARCHAR(DATEADD('month', -6, DATE_TRUNC('month', '2026-07-31'::DATE)), 'YYYYMMDD'),
       -1500000.00, -150000.00, 'AUD', 'S1', 'SUP000001', 'AR_INVOICE',
       'Backdated revenue adjustment - posted after BAS lodgement';

In [ ]:
-- 3. Force the Dynamic Table to pick it up now rather than waiting for
--    its one-hour target lag. TARGET_LAG is a freshness objective;
--    this is how you demand freshness immediately.
ALTER DYNAMIC TABLE CURATED.GL_HARMONISED REFRESH;

In [ ]:
-- 4. Re-run the reconciliation for that entity and period only.
SELECT ENTITY_NAME, TAX_PERIOD, COMPUTED_1A, LODGED_1A,
       VARIANCE_NET, VARIANCE_PCT, RECON_STATUS
FROM TAX.V_GST_VS_BAS
WHERE ENTITY_ID  = 'ENT001'
  AND TAX_PERIOD = DATEADD('month', -6, DATE_TRUNC('month', '2026-07-31'::DATE));

That entity-period was `MATCHED` at $0.00 four minutes ago. It is now `INVESTIGATE`, carrying roughly $150,000 of GST on sales that the lodged BAS does not contain. Nothing was recalculated by hand and no report was regenerated: one row landed in a raw table, the Dynamic Table refreshed, and the reconciliation told a different and correct story.

> ⚠️ **The lodged figure has not changed — and cannot**
>
> This is the distinction that matters to a tax practitioner. The BAS you lodged is a historical fact. The ledger underneath it is not: it keeps moving. Any reconciliation that recomputes from today's ledger and compares against a return lodged months ago will drift, and the drift is not an error — it is the adjustment. What you need is the ability to see both positions.

### The as-lodged position, in one clause

The auditor's question is now live: *show me the ledger exactly as it stood when we lodged.* The answer is not a backup, a restore, a snapshot table somebody remembered to populate, or a ticket to IT. It is a clause.

In [ ]:
SELECT COUNT(*)                    AS LINES_AS_AT_THAT_MOMENT,
       ROUND(SUM(BOOKED_TAX_AMOUNT), 2) AS BOOKED_TAX_AS_LODGED
FROM CURATED.GL_HARMONISED AT(TIMESTAMP => $pre_adjustment)
WHERE ENTITY_ID  = 'ENT001'
  AND TAX_PERIOD = DATEADD('month', -6, DATE_TRUNC('month', '2026-07-31'::DATE));

That is the ledger as it stood at the moment you named, before the adjustment existed. The row count is one lower and the booked tax total is the as-lodged figure — the number that ties to the BAS.

Three ways to name a point in time. `AT(TIMESTAMP => ...)` is the one to reach for: you captured `$pre_adjustment` before you posted anything, so the query means exactly what it says regardless of how long the lab takes. `AT(OFFSET => -3600)` counts seconds back from now, which sounds convenient and is the classic trap — if the object is younger than the offset, or you pause for coffee between cells, it either errors or silently returns the wrong moment. `BEFORE(STATEMENT => '<query_id>')` is the most precise of the three: the state immediately before one specific statement, which is what you want when an auditor asks about one specific posting.

Notice what you did **not** do: you did not restore a database, you did not stand up a copy, you did not go looking for last quarter's extract, and you did not ask anyone's permission. You added eighteen characters to a query you already had.

### Freeze the evidence, break it, get it back

Time Travel answers the question in the moment. The auditor wants something they can come back to next month, so freeze it: a clone of the ledger as it stood before the adjustment, named for the date it was taken, in a schema that exists for exactly this purpose.

In [ ]:
-- An immutable, named snapshot of the whole 2,400,000-line ledger
-- as it stood at the moment before the adjustment was posted.
CREATE OR REPLACE TABLE AUDIT.GL_AS_LODGED_20260803 CLONE CURATED.GL_HARMONISED AT(TIMESTAMP => $pre_adjustment);

In [ ]:
SELECT COUNT(*) AS SNAPSHOT_LINES FROM AUDIT.GL_AS_LODGED_20260803;

In [ ]:
-- Now the accident everybody has had. Wrong window, wrong object.
DROP TABLE AUDIT.GL_AS_LODGED_20260803;

In [ ]:
-- And the recovery. Within the retention window, one word.
UNDROP TABLE AUDIT.GL_AS_LODGED_20260803;

In [ ]:
SELECT COUNT(*) AS STILL_HERE FROM AUDIT.GL_AS_LODGED_20260803;

> ✅ **Zero-copy cloning — what just happened, and what it cost**
>
> You took an immutable snapshot of a **2,400,000-row ledger**. It completed instantly and it consumed **no storage**, because a clone copies metadata rather than data — the two objects share the same underlying micro-partitions until one of them diverges, and you pay only for what subsequently changes. Worth costing that the way you would have to today: how long to extract 2.4 million rows, where they would be written, who approves the disk, and who remembers the extract exists in eighteen months when the audit actually lands.

The wider point is that none of this required a backup strategy, a restore procedure, or a conversation with IT. There was no data protection design decision to make in advance, no retention policy to negotiate, and no recovery runbook to test. Time Travel, cloning and `UNDROP` are properties of the platform — they are true of every table in the account, whether or not anybody planned for them.

### The visual close, with no new SQL

Re-run the reconciliation, then read it as a picture rather than a grid. There is no charting library to install and no new query to write.

In [ ]:
SELECT ENTITY_NAME,
       ROUND(SUM(ABS(VARIANCE_NET)), 2) AS TOTAL_ABS_VARIANCE
FROM TAX.V_GST_VS_BAS
GROUP BY ENTITY_NAME
ORDER BY TOTAL_ABS_VARIANCE DESC;

1. Run the query and wait for the grid to appear.
2. Click the **Chart** tab, immediately beside **Results** above the output pane.
3. Set the chart type to **Bar** in the panel on the right.
4. Set the X axis to `ENTITY_NAME` and the Y axis to `TOTAL_ABS_VARIANCE`.

The distribution is the finding. Variance is not spread evenly across the group — it concentrates in a handful of entities, and those are where the tax team should spend its week. This is the chart you put in front of the client, built from a query you were running anyway, and it takes four clicks.

> 💡 **Why this matters more than it looks**
>
> Most analysis dies between the query and the audience. A number in a results grid is not a finding until somebody can see the shape of it. That the platform charts a result set without a tool, an export or a licence means the distance from question to defensible picture is measured in clicks — and the optional Streamlit step turns the same two views into a shareable application without leaving Snowflake.

### What the environment now holds

The `RAW_DATA` and `TAX` rule tables came from setup. Everything from `CURATED.GL_HARMONISED` down you built in this lab.

| Object | Type | Rows | Purpose |
|---|---|---|---|
| `RAW_DATA.ERP_SAP_GL` | TABLE | 1,200,000 | SAP general ledger, as landed. |
| `RAW_DATA.ERP_ORACLE_GL` | TABLE | 800,000 | Oracle general ledger, as landed. |
| `RAW_DATA.ERP_LEGACY_AP` | TABLE | 400,000 | Legacy AP system, text dates, GST-inclusive amounts. |
| `RAW_DATA.DIM_LEGAL_ENTITY` | TABLE | 14 | Group entities and the three ERP identifier schemes. |
| `RAW_DATA.DIM_ACCOUNT_MAPPING` | TABLE | 180 | Native account code to group account and supply category. |
| `RAW_DATA.DIM_SUPPLIER` | TABLE | 2,000 | Supplier ABN, GST registration and country. |
| `RAW_DATA.FCT_BAS_LODGED` | TABLE | 532 | What was actually lodged with the ATO. |
| `RAW_DATA.DOC_TAX_INVOICE` | TABLE | 300 | Unstructured invoice text — Lab 2 uses this. |
| `TAX.TAX_RULE_DETERMINATION` | TABLE | 16 | The decision table: *the first rule set of many — see the end of Step 4*. |
| `TAX.TAX_RULE_RATE` | TABLE | 19 | Effective-dated rates, part of the same rules layer. |
| `CURATED.GL_HARMONISED` | **DYNAMIC TABLE** | 2,400,000 | One governed ledger. No tax logic. Keeps itself current. |
| `TAX.V_GST_DETERMINED` | VIEW | 2,400,000 | Treatment, rate and variance per line, with rule lineage. |
| `TAX.V_GST_VS_BAS` | VIEW | 532 | Computed against lodged, by entity and period. |
| `TAX.V_GST_EXCEPTIONS` | VIEW | 253,394 | Risk-ranked exception register, seven rules. |

Four objects of your own creation on top of the source data, one of which is a Dynamic Table and three of which are views — and a reconciliation that ties to the cent for 500 of 532 entity-periods. There are three optional steps below: a Streamlit control tower over these two views, a scenario-modelling exercise on a cloned database, and a secure share of the audit snapshot. Lab 2 takes the same objects into Cortex AI — classification, extraction and a Cortex Agent your client can ask questions of in English.

> ⚠️ **There is deliberately no cleanup in Lab 1**
>
> Do not drop `GSI_TAX_WORKSHOP` or any object inside it. Lab 2 builds directly on `CURATED.GL_HARMONISED`, `TAX.V_GST_DETERMINED`, `TAX.V_GST_VS_BAS` and `TAX.V_GST_EXCEPTIONS`. The only thing the optional steps create and then remove is the scenario database in Step 7, which is dropped inside that step because it exists purely to be thrown away.

> **What the tax practice can do now**
>
> - [x] see all three ERP extracts side by side, unmodified — *Step 1*
> - [x] show exactly why the group GST position cannot be answered today — *Step 2*
> - [x] query 2,400,000 lines as one governed ledger that maintains itself — *Step 3*
> - [x] read the entire determination logic without reading SQL — *Step 4*
> - [x] reconcile the computed position against what was actually lodged — *Step 5*
> - [x] reproduce any prior filing on demand, and prove nothing moved — *Step 5*

---

## Step 6 of 8 — Streamlit tax control tower

*Optional extension — the core flow is complete without this.*

*~12 min · OPTIONAL* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-6)

**⭐ RECOMMENDED FIRST.** Of the three optional steps, do this one if you only do one. Everything you have built so far lives in a results grid. A results grid is fine for you and useless to the tax manager who has to work a month of exceptions, and worse than useless in front of a client. This step turns the same two views into a screen the practice can actually run a review from — written in Python, running inside Snowflake, with no infrastructure and no connection string.

- Build a single-page GST control tower over `TAX.V_GST_VS_BAS` and `TAX.V_GST_EXCEPTIONS`.
- Use the session that already exists rather than authenticating again.
- Cache every query so filtering the page does not re-scan the ledger.
- See why every query in the app aggregates or bounds its result before it returns.

### Three decisions worth understanding before you paste it

**One file, no project.** The whole dashboard — layout, theme, queries and charts — is a single `.py` file. There is no `requirements.txt`, no `.streamlit/config.toml` and nothing to build. That matters for handing it on: a colleague can copy one file into their own account and it runs. The dark theme is injected as CSS inside the app for exactly that reason.

**Aggregate in Snowflake, render the answer.** The two views sit on 2.4 million ledger lines and 253,394 flagged exceptions. Every query in the app either aggregates server side and returns tens of rows, or ranks by `RISK_SCORE` and returns 150. Nobody reads a quarter of a million rows, so nothing is gained by shipping them to a browser.

**Draw the visuals in HTML, not a plotting library.** The heatmap and the score distribution are hand-built with CSS grid and inline SVG. Partly because `matplotlib` is not in the Streamlit in Snowflake warehouse runtime — anything depending on it, including pandas' `Styler.background_gradient`, fails at render time rather than at import. Partly because it keeps the app to one file with no packages to declare.

### Create the app in Snowsight

1. In the left navigation choose **Projects**, then **Streamlit**.
2. Click **+ Streamlit App**.
3. Name it `GST Control Tower`.
4. Set **App location** to database `GSI_TAX_WORKSHOP` and schema `APPS`.
5. Set **App warehouse** to `GSI_TAX_WH`.
6. Click **Create**, delete the sample code in the editor, paste the block below, and click **Run**.

The app runs unmodified. Nothing to edit, nothing to configure, no packages to add — Streamlit and the Snowpark library are already in the environment.

### The control tower app

In [ ]:
# The control tower app. This is the file to paste into Projects -> Streamlit
# (add st.set_page_config back at the top when you do). Running the cell here
# renders the dashboard inline, but its theme assumes a full page, so the
# deployed app is the one to look at.
import streamlit as st
import bisect
import html

import pandas as pd
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# =====================================================================
# THEME
#
# One dark theme, defined once, applied with injected CSS so the whole
# app stays a SINGLE file - no .streamlit/config.toml to copy alongside it.
#
# Because the background is forced dark, every foreground colour is set
# explicitly too. Streamlit's own widgets default to the light theme's dark
# text, which would be invisible here, so widget labels, inputs and dropdowns
# are all given an explicit light colour below.
# =====================================================================
BG      = "#0B1220"   # page
PANEL   = "#111C2E"   # cards
EDGE    = "#1E2D45"   # borders
INK     = "#E6EDF6"   # primary text
MUTED   = "#8A9BB0"   # secondary text
BLUE    = "#29B5E8"   # Snowflake accent
RED     = "#F76B7C"   # investigate / high
AMBER   = "#E8B23A"   # review / medium
GREEN   = "#4ED8A0"   # matched / clean

CSS = """
<style>
:root {
  --bg:#0B1220; --panel:#111C2E; --edge:#1E2D45;
  --ink:#E6EDF6; --muted:#8A9BB0;
  --blue:#29B5E8; --red:#F76B7C; --amber:#E8B23A; --green:#4ED8A0;
}
/* page + base text -------------------------------------------------- */
.stApp, [data-testid="stAppViewContainer"] { background: var(--bg); }
[data-testid="stHeader"] { background: transparent; }
.block-container { padding: 1.1rem 1.4rem 3rem; max-width: 1500px; }
.stApp, .stApp p, .stApp span, .stApp div, .stApp label {
  color: var(--ink);
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
}
/* every Streamlit widget label, explicitly light ---------------------- */
[data-testid="stWidgetLabel"], [data-testid="stWidgetLabel"] p,
.stSlider label, .stSelectbox label, .stTextInput label, .stMultiSelect label {
  color: var(--muted) !important;
  font-size: .68rem !important; font-weight: 700 !important;
  letter-spacing: .09em !important; text-transform: uppercase !important;
}
/* Inputs: dark field, light text.
   Streamlit changed its widget internals: older builds (which is what the
   Snowflake warehouse runtime ships) render selects with BaseWeb and expose
   data-baseweb="select"; newer builds use react-aria and expose neither. We
   target both, plus the stable data-testid wrappers, so the dashboard stays
   dark whichever build it lands on. The wrapper is what carries the default
   light fill, so styling only the <input> leaves a white box around it. */
.stTextInput input, .stNumberInput input,
[data-testid="stTextInputRootElement"],
[data-testid="stTextInputRootElement"] > div {
  background: #0D1728 !important;
  color: var(--ink) !important;
  -webkit-text-fill-color: var(--ink) !important;
  border-color: var(--edge) !important;
  border-radius: 8px !important;
}
.stTextInput input::placeholder { color: #5E7189 !important; opacity: 1 !important; }
/* select shell: BaseWeb (older) and react-aria (newer) */
[data-baseweb="select"] > div,
[data-testid="stSelectbox"] [role="group"],
[data-testid="stMultiSelect"] [role="group"],
.react-aria-ComboBox [role="group"] {
  background: #0D1728 !important; border-color: var(--edge) !important;
  color: var(--ink) !important; border-radius: 8px !important;
}
[data-baseweb="select"] div, [data-baseweb="select"] span,
[data-testid="stSelectbox"] input, [data-testid="stMultiSelect"] input,
[data-testid="stSelectbox"] [role="combobox"] {
  color: var(--ink) !important; -webkit-text-fill-color: var(--ink) !important;
  background: transparent !important;
}
/* dropdown panel */
[data-baseweb="popover"], [data-baseweb="menu"], [role="listbox"],
.react-aria-Popover, .react-aria-ListBox {
  background: #0D1728 !important; border: 1px solid var(--edge) !important;
}
[role="option"], .react-aria-ListBoxItem {
  background: #0D1728 !important; color: var(--ink) !important;
}
[role="option"]:hover, .react-aria-ListBoxItem[data-focused] { background: #16243A !important; }
[data-baseweb="tag"] { background: #16243A !important; color: var(--ink) !important; }
/* slider ------------------------------------------------------------- */
.stSlider [data-baseweb="slider"] div[role="slider"],
.stSlider [role="slider"] { background: var(--blue) !important; }
.stSlider [data-testid="stTickBar"], .stSlider [data-testid="stTickBarMin"],
.stSlider [data-testid="stTickBarMax"] { color: var(--muted) !important; background: transparent !important; }
/* kill Streamlit chrome we are replacing ----------------------------- */
[data-testid="stToolbar"], footer, #MainMenu { visibility: hidden; }
hr { border-color: var(--edge); }

/* ---------------- our own components ------------------------------- */
.topbar {
  display:flex; align-items:center; gap:14px;
  background:linear-gradient(135deg,#12243C 0%,#0E1A2C 55%,#141B33 100%);
  border:1px solid var(--edge); border-radius:14px; padding:14px 18px; margin-bottom:14px;
}
.logo {
  width:38px;height:38px;border-radius:10px;flex:0 0 38px;
  background:linear-gradient(135deg,var(--blue),#1273A8);
  display:flex;align-items:center;justify-content:center;font-size:19px;
}
.t-title { font-size:1.28rem; font-weight:800; letter-spacing:-.02em; line-height:1.15; }
.t-meta  { font-size:.76rem; color:var(--muted); margin-top:3px; }
.dot { height:7px;width:7px;border-radius:50%;display:inline-block;margin-right:5px;
       background:var(--green); box-shadow:0 0 0 3px rgba(78,216,160,.16); }

.kpis { display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin-bottom:14px; }
.kpi { background:var(--panel); border:1px solid var(--edge); border-radius:14px;
       padding:15px 17px 13px; position:relative; overflow:hidden; }
.kpi-l { font-size:.65rem; font-weight:700; letter-spacing:.1em; text-transform:uppercase; color:var(--muted); }
.kpi-v { font-size:2.15rem; font-weight:800; letter-spacing:-.03em; line-height:1.12; margin:5px 0 2px; }
.kpi-s { font-size:.74rem; color:var(--muted); min-height:2.5em; }
.kpi-track { position:absolute; left:17px; right:17px; bottom:9px; height:3px;
             background:#1B2942; border-radius:3px; overflow:hidden; }
.kpi-fill { height:100%; border-radius:3px; animation:grow 1.05s cubic-bezier(.22,1,.36,1) both; }
@keyframes grow { from { width:0 } }

.card { background:var(--panel); border:1px solid var(--edge); border-radius:14px;
        padding:15px 17px 17px; height:100%; }
.card-h { display:flex; justify-content:space-between; align-items:baseline; gap:14px; margin-bottom:14px; }
.card-t { font-size:.95rem; font-weight:700; white-space:nowrap; }
.card-n { font-size:.71rem; color:var(--muted); text-align:right; }

.brow { margin-bottom:13px; }
.brow-top { display:flex; justify-content:space-between; align-items:baseline; font-size:.82rem; }
.brow-track { height:8px; background:#1B2942; border-radius:5px; margin-top:6px; overflow:hidden; }
.brow-fill { height:100%; border-radius:5px; animation:grow 1.15s cubic-bezier(.22,1,.36,1) both; }

table.tt { width:100%; border-collapse:collapse; font-size:.79rem; }
table.tt th { text-align:left; font-size:.63rem; letter-spacing:.09em; text-transform:uppercase;
              color:var(--muted); font-weight:700; padding:0 9px 8px; border-bottom:1px solid var(--edge); }
table.tt td { padding:8px 9px; border-bottom:1px solid #16223A; color:var(--ink); }
table.tt tr:hover td { background:#16243A; }
.num { font-variant-numeric:tabular-nums; text-align:right;
       font-family:ui-monospace,SFMono-Regular,Menlo,monospace; }
.pill { display:inline-block; padding:2px 9px; border-radius:999px;
        font-size:.66rem; font-weight:700; letter-spacing:.03em; }

.hm { display:grid; gap:2px; }
.hm-c { height:19px; border-radius:3px; }
.hm-c:hover { outline:1.5px solid var(--ink); z-index:6; }

/* Instant hover tooltip, shared by the heatmap cells and the histogram buckets.
   The browser's native title attribute waits about a second and renders in OS
   chrome, so in a room people conclude hover is broken before it ever appears. */
[data-tip] { position:relative; }
[data-tip]:hover::after {
  content:attr(data-tip); position:absolute; bottom:calc(100% + 7px); left:50%;
  transform:translateX(-50%); background:#050A14; color:var(--ink);
  border:1px solid var(--edge); padding:6px 10px; border-radius:6px;
  font-size:.7rem; white-space:nowrap; z-index:30; pointer-events:none;
  box-shadow:0 8px 22px rgba(0,0,0,.6);
}
/* Clamp the first and last columns. A centred tooltip on the final period runs
   past the viewport, and Streamlit's app container is overflow:hidden, so it
   would be cut in half rather than just overhanging. */
[data-tip].tip-r:hover::after { left:0; transform:none; }
[data-tip].tip-l:hover::after { left:auto; right:0; transform:none; }

/* Transparent hit targets laid over the histogram, one per bucket, so the bars
   get the same instant tooltip as the heatmap without rewriting the SVG. */
.hx { position:relative; }
.hx-c { position:absolute; top:0; border-radius:4px; }
.hx-c:hover { background:rgba(230,237,246,.06); }
.hm-lbl { font-size:.7rem; color:var(--muted); padding-right:9px; text-align:right;
          white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)


# =====================================================================
# DATA
# Every query aggregates or bounds its result. Nothing returns a row per
# ledger line: the register is 253,394 rows and the browser does not want them.
# =====================================================================
@st.cache_data(ttl=600)
def recon_summary():
    return session.sql("""
        SELECT RECON_STATUS, COUNT(*) AS N, ROUND(SUM(ABS(VARIANCE_NET)),2) AS ABS_VAR
        FROM TAX.V_GST_VS_BAS GROUP BY 1
    """).to_pandas()


@st.cache_data(ttl=600)
def ledger_scale():
    """Header and KPI figures, read live rather than hardcoded.

    Lab 1 posts a correcting entry, which moves the ledger count off the round
    2,400,000 and shifts one entity-period out of MATCHED. A hardcoded number
    would quietly start lying the moment a participant runs that step.
    """
    return session.sql("""
        SELECT (SELECT COUNT(*) FROM CURATED.GL_HARMONISED) AS LINES,
               COUNT(DISTINCT ENTITY_ID)                    AS ENTITIES,
               COUNT(DISTINCT TAX_PERIOD)                   AS PERIODS
        FROM TAX.V_GST_VS_BAS
    """).to_pandas().iloc[0]


@st.cache_data(ttl=600)
def exception_summary():
    return session.sql("""
        SELECT COUNT(*) AS FLAGGED, SUM(IFF(MAX_SEVERITY='HIGH',1,0)) AS HIGH,
               ROUND(AVG(RISK_SCORE),2) AS AVG_RISK
        FROM TAX.V_GST_EXCEPTIONS
    """).to_pandas().iloc[0]


@st.cache_data(ttl=600)
def score_histogram():
    return session.sql("""
        SELECT FLOOR(RISK_SCORE/10)*10 AS BUCKET, MAX_SEVERITY, COUNT(*) AS N
        FROM TAX.V_GST_EXCEPTIONS GROUP BY 1,2 ORDER BY 1
    """).to_pandas()


@st.cache_data(ttl=600)
def entities():
    return session.sql(
        "SELECT DISTINCT ENTITY_NAME FROM TAX.V_GST_EXCEPTIONS ORDER BY 1"
    ).to_pandas()["ENTITY_NAME"].tolist()


@st.cache_data(ttl=600)
def severities():
    """Severity levels that actually occur, ordered high to low.

    Hardcoding HIGH/MEDIUM/LOW offered a LOW filter that returns an empty table,
    because the rule set only ever lands on MEDIUM or HIGH. A dead filter option
    reads as a broken app. Reading the levels from the data means the control
    always matches reality, and a LOW option appears by itself if the rules ever
    start producing one.
    """
    have = session.sql(
        "SELECT DISTINCT MAX_SEVERITY FROM TAX.V_GST_EXCEPTIONS"
    ).to_pandas()["MAX_SEVERITY"].tolist()
    return [s for s in ("HIGH", "MEDIUM", "LOW") if s in have]


@st.cache_data(ttl=600)
def heatmap():
    return session.sql("""
        SELECT ENTITY_NAME, TO_VARCHAR(TAX_PERIOD,'YYYY-MM') AS P, COUNT(*) AS N
        FROM TAX.V_GST_EXCEPTIONS WHERE TAX_PERIOD IS NOT NULL
        GROUP BY 1,2 ORDER BY 1,2
    """).to_pandas()


@st.cache_data(ttl=600)
def worklist(min_score, sev, entity, search, limit):
    where = [f"RISK_SCORE >= {int(min_score)}"]
    if sev != "All":
        where.append(f"MAX_SEVERITY = '{sev}'")
    if entity != "All entities":
        where.append(f"ENTITY_NAME = '{entity}'")
    if search:
        safe = search.replace("'", "''").upper()
        where.append(f"(UPPER(SUPPLIER_NAME) LIKE '%{safe}%' OR UPPER(GL_LINE_ID) LIKE '%{safe}%')")
    clause = "WHERE " + " AND ".join(where)
    return session.sql(f"""
        SELECT GL_LINE_ID, ENTITY_NAME, SUPPLIER_NAME, SUPPLY_CATEGORY,
               TAX_TREATMENT, APPLIED_RULE_ID, TAX_VARIANCE, EXCEPTION_CODES,
               MAX_SEVERITY, RISK_SCORE
        FROM TAX.V_GST_EXCEPTIONS {clause}
        ORDER BY RISK_SCORE DESC, ABS(TAX_VARIANCE) DESC
        LIMIT {int(limit)}
    """).to_pandas(), clause


@st.cache_data(ttl=600)
def filtered_totals(clause):
    return session.sql(f"""
        SELECT COUNT(*) AS N, ROUND(SUM(ABS(COALESCE(TAX_VARIANCE,0))),2) AS ABS_VAR
        FROM TAX.V_GST_EXCEPTIONS {clause}
    """).to_pandas().iloc[0]


def txt(v):
    """NULL-safe cell text for the hand-rolled table.

    Snowflake NULLs arrive as float nan through to_pandas(), and dropping one
    straight into an f-string renders the literal string 'nan'. Unmapped GL
    accounts genuinely have no supply category, so this is the common case
    rather than an edge one. Values are escaped because they land in raw HTML.
    """
    if v is None or pd.isna(v):
        return "<span style='opacity:.5'>&mdash;</span>"
    return html.escape(str(v))


def money(v):
    v = float(v or 0)
    for cut, suf in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
        if abs(v) >= cut:
            return f"${v / cut:,.1f}{suf}"
    return f"${v:,.0f}"


# =====================================================================
# HEADER
# =====================================================================
rs = recon_summary()
sc = ledger_scale()
ex = exception_summary()
total_periods = int(rs["N"].sum())
investigate = int(rs.loc[rs.RECON_STATUS == "INVESTIGATE", "N"].sum())
abs_var = float(rs["ABS_VAR"].sum())
flagged = int(ex["FLAGGED"])
stamp = datetime.now(timezone.utc).strftime("%d %b %Y %H:%M UTC")

st.markdown(f"""
<div class="topbar">
  <div class="logo">&#128202;</div>
  <div style="flex:1">
    <div class="t-title">GST Control Tower</div>
    <div class="t-meta"><span class="dot"></span>Live &nbsp;&middot;&nbsp;
      GSI_TAX_WORKSHOP.TAX &nbsp;&middot;&nbsp; {int(sc['LINES']):,} ledger lines
      &nbsp;&middot;&nbsp; Streamlit on Snowflake &nbsp;&middot;&nbsp; {stamp}</div>
  </div>
</div>
""", unsafe_allow_html=True)


def kpi(label, value, sub, colour, pct):
    return (f'<div class="kpi"><div class="kpi-l">{label}</div>'
            f'<div class="kpi-v" style="color:{colour}">{value}</div>'
            f'<div class="kpi-s">{sub}</div>'
            f'<div class="kpi-track"><div class="kpi-fill" '
            f'style="width:{pct:.1f}%;background:{colour}"></div></div></div>')


st.markdown(
    '<div class="kpis">'
    + kpi("Entity-periods reconciled", f"{total_periods:,}",
          f"{int(sc['ENTITIES'])} entities across {int(sc['PERIODS'])} complete months",
          BLUE, 100)
    + kpi("Investigate", f"{investigate:,}",
          "variance above the $1,000 threshold", RED,
          100 * investigate / max(total_periods, 1))
    + kpi("Flagged ledger lines", f"{flagged:,}",
          f"avg risk score {float(ex['AVG_RISK']):.1f}", AMBER,
          100 * flagged / max(int(sc["LINES"]), 1))
    + kpi("Absolute variance", money(abs_var),
          "computed against lodged BAS", GREEN, 100)
    + "</div>", unsafe_allow_html=True)


# =====================================================================
# FILTERS - native widgets, explicitly styled above so the text is legible
# =====================================================================
f1, f2, f3, f4 = st.columns([1, 1.4, 1.6, 1.1])
sev = f1.selectbox("Severity", ["All"] + severities())
entity = f2.selectbox("Entity", ["All entities"] + entities())
min_score = f3.slider("Minimum risk score", 0, 100, 0, 5)
search = f4.text_input("Search supplier or line", placeholder="e.g. Ridgeline")

rows, clause = worklist(min_score, sev, entity, search, 150)
tot = filtered_totals(clause)

# =====================================================================
# PANELS
# =====================================================================
left, right = st.columns([1, 1.25])

SEV_COLOUR = {"MATCHED": GREEN, "REVIEW": AMBER, "INVESTIGATE": RED}

with left:
    bars = ""
    top = int(rs["N"].max())
    for status in ("MATCHED", "REVIEW", "INVESTIGATE"):
        n = int(rs.loc[rs.RECON_STATUS == status, "N"].sum())
        c = SEV_COLOUR[status]
        bars += (
            f'<div class="brow"><div class="brow-top">'
            f'<span><span class="dot" style="background:{c};box-shadow:none"></span>'
            f'{status.title()}</span>'
            f'<span><b>{n:,}</b> <span style="color:{MUTED}">&middot; '
            f'{100 * n / max(total_periods, 1):.1f}%</span></span></div>'
            f'<div class="brow-track"><div class="brow-fill" '
            f'style="width:{100 * n / max(top, 1):.1f}%;background:{c}"></div></div></div>')
    st.markdown(
        f'<div class="card"><div class="card-h">'
        f'<span class="card-t">Reconciliation against lodged BAS</span>'
        f'<span class="card-n">{total_periods:,} entity-periods</span></div>'
        f'{bars}<div class="card-n" style="margin-top:6px">Matched means variance of '
        f'exactly $0.00 &mdash; not within tolerance, nil.</div></div>',
        unsafe_allow_html=True)

with right:
    hist = score_histogram()
    grid = hist.pivot_table(index="BUCKET", columns="MAX_SEVERITY",
                            values="N", aggfunc="sum").fillna(0)
    buckets = list(range(0, 100, 10))
    peak = max(int(grid.sum(axis=1).max()), 1)
    W, H, PAD = 560, 250, 26
    bw = (W - PAD) / len(buckets)
    svg = [f'<svg viewBox="0 0 {W} {H + 34}" width="100%" height="{H + 34}">']
    for gl in (0, 0.5, 1):
        y = H - gl * (H - PAD)
        svg.append(f'<line x1="{PAD}" y1="{y:.0f}" x2="{W}" y2="{y:.0f}" '
                   f'stroke="#1B2942" stroke-width="1"/>')
        svg.append(f'<text x="0" y="{y - 4:.0f}" fill="{MUTED}" font-size="10">'
                   f'{int(peak * gl):,}</text>')
    for i, b in enumerate(buckets):
        x = PAD + i * bw
        stack = 0.0
        for s, c in (("LOW", GREEN), ("MEDIUM", AMBER), ("HIGH", RED)):
            n = float(grid.loc[b, s]) if (b in grid.index and s in grid.columns) else 0.0
            if n <= 0:
                continue
            h = (n / peak) * (H - PAD)
            stack += h
            svg.append(f'<rect x="{x + 3:.1f}" y="{H - stack:.1f}" width="{bw - 6:.1f}" '
                       f'height="{h:.1f}" fill="{c}" rx="2"><title>score {b}-{b + 9} '
                       f'&middot; {s} &middot; {int(n):,} lines</title></rect>')
        svg.append(f'<text x="{x + bw / 2:.0f}" y="{H + 15}" fill="{MUTED}" font-size="10" '
                   f'text-anchor="middle">{b}</text>')
    svg.append(f'<text x="{W / 2:.0f}" y="{H + 31}" fill="{MUTED}" font-size="10" '
               f'text-anchor="middle">risk score</text></svg>')
    # Legend covers only the severities present in the chart. Listing a level
    # with no bars invites the reader to hunt for a colour that is not there.
    legend = "".join(
        f'<span style="margin-left:11px"><span class="dot" style="background:{c};'
        f'box-shadow:none"></span><span style="color:{MUTED};font-size:.71rem">{s.title()}'
        f'</span></span>'
        for s, c in (("HIGH", RED), ("MEDIUM", AMBER), ("LOW", GREEN))
        if s in grid.columns and float(grid[s].sum()) > 0)
    # One transparent hit target per bucket, laid over the SVG. Carries the whole
    # severity split in a single tooltip, which reads better than hovering each
    # stacked segment in turn.
    ov = ""
    for i, b in enumerate(buckets):
        parts, tot_b = [], 0
        for s in ("HIGH", "MEDIUM", "LOW"):
            n = int(grid.loc[b, s]) if (b in grid.index and s in grid.columns) else 0
            tot_b += n
            if n:
                parts.append(f"{s.title()} {n:,}")
        tip = f"score {b}&ndash;{b + 9} &middot; {tot_b:,} lines"
        if parts:
            tip += " &middot; " + " &middot; ".join(parts)
        edge = "tip-r" if i <= 1 else ("tip-l" if i >= len(buckets) - 2 else "")
        ov += (f'<div class="hx-c {edge}" style="left:{100 * (PAD + i * bw) / W:.2f}%;'
               f'width:{100 * bw / W:.2f}%;height:{100 * H / (H + 34):.2f}%" '
               f'data-tip="{tip}"></div>')
    st.markdown(
        f'<div class="card" style="overflow:visible"><div class="card-h">'
        f'<span class="card-t">Risk score distribution</span>'
        f'<span class="card-n">10-point bins{legend}</span>'
        f'</div><div class="hx">{"".join(svg)}{ov}</div></div>', unsafe_allow_html=True)


# =====================================================================
# HEATMAP - entity x period, shaded with our own scale (no matplotlib)
# =====================================================================
hm = heatmap()
piv = hm.pivot(index="ENTITY_NAME", columns="P", values="N").fillna(0).astype(int)
hi = max(int(piv.to_numpy().max()), 1)
# Endpoints for the legend. The shading itself is rank-based (see below).
lo = int(piv.to_numpy().min())


def rag(t):
    """Green-amber-red ramp for a 0..1 position in the observed range.

    The scale is comparative, not absolute. Every entity-period carries a few
    hundred flagged lines, so green means "quietest month in this population",
    never "clean" - which is why the caption says fewest rather than good. A
    reviewer who reads green as clean would draw exactly the wrong conclusion.
    """
    stops = ((0.0, (78, 216, 160)), (0.5, (232, 178, 58)), (1.0, (247, 107, 124)))
    for (t0, c0), (t1, c1) in zip(stops, stops[1:]):
        if t <= t1:
            f = (t - t0) / (t1 - t0)
            return tuple(round(a + (b - a) * f) for a, b in zip(c0, c1))
    return stops[-1][1]


periods = list(piv.columns)
# Shade by rank within the grid, not by linear position in the range. The
# distribution is left-skewed - the median entity-period sits at 0.73 of the
# range - so a linear ramp puts three quarters of the grid in the red and the
# whole thing reads as one alarming block with no signal in it. Ranking spends
# the full green-to-red spread on the ordering that matters. The caption says
# "by rank" for exactly this reason: green is the quietest cell here, not a
# clean one, and a reviewer must not read it as clean.
flat = sorted(int(v) for v in piv.to_numpy().ravel())


def pct(n):
    """Position of n in the observed distribution, 0..1, ties averaged."""
    a, b = bisect.bisect_left(flat, n), bisect.bisect_right(flat, n)
    return ((a + b - 1) / 2) / max(len(flat) - 1, 1)


median = flat[len(flat) // 2]
cells = ""
for ent in piv.index:
    cells += f'<div class="hm-lbl">{html.escape(str(ent))}</div>'
    for i, p in enumerate(periods):
        n = int(piv.loc[ent, p])
        r, g, b = rag(pct(n))
        edge = "tip-r" if i <= 1 else ("tip-l" if i >= len(periods) - 2 else "")
        cells += (f'<div class="hm-c {edge}" style="background:rgb({r},{g},{b})" '
                  f'data-tip="{html.escape(str(ent))} &middot; {p} &middot; '
                  f'{n:,} flagged lines"></div>')
ramp = "".join(
    f'<span style="display:inline-block;width:15px;height:9px;background:'
    f'rgb{rag(i / 6)};border-radius:2px;margin-left:2px"></span>' for i in range(7))
st.markdown(
    f'<div class="card" style="margin-top:14px;overflow:visible">'
    f'<div class="card-h"><span class="card-t">Flagged lines by entity and period'
    f'</span><span class="card-n">{len(piv.index)} entities &times; {len(periods)} '
    f'periods &middot; hover a cell for detail<br>fewest {lo:,}{ramp} most {hi:,}'
    f' &middot; shaded by rank &middot; median {median:,}</span></div>'
    f'<div class="hm" style="grid-template-columns:150px repeat({len(periods)},1fr)">'
    f'{cells}</div></div>', unsafe_allow_html=True)


# =====================================================================
# WORKLIST - rendered as our own table, so no st.dataframe theming fight
# =====================================================================
PILL = {"HIGH": (RED, "rgba(247,107,124,.14)"),
        "MEDIUM": (AMBER, "rgba(232,178,58,.14)"),
        "LOW": (GREEN, "rgba(78,216,160,.14)")}
body = ""
for _, r in rows.iterrows():
    fg, bg = PILL.get(r.MAX_SEVERITY, (MUTED, "rgba(138,155,176,.14)"))
    var = float(r.TAX_VARIANCE or 0)
    body += (
        f"<tr><td style='font-family:ui-monospace,Menlo,monospace;color:{MUTED}'>"
        f"{txt(r.GL_LINE_ID)}</td><td>{txt(r.ENTITY_NAME)}</td><td>{txt(r.SUPPLIER_NAME)}</td>"
        f"<td style='color:{MUTED}'>{txt(r.SUPPLY_CATEGORY)}</td><td>{txt(r.TAX_TREATMENT)}</td>"
        f"<td class='num' style='color:{MUTED}'>{txt(r.APPLIED_RULE_ID)}</td>"
        # Direction matters to a tax reviewer: under-declared GST is ATO
        # exposure, over-declared is the firm's own money sitting with the ATO.
        f"<td class='num' style='color:{RED if var < 0 else AMBER}'>{var:,.2f}</td>"
        f"<td style='color:{MUTED};font-size:.72rem'>{txt(r.EXCEPTION_CODES)}</td>"
        f"<td><span class='pill' style='color:{fg};background:{bg}'>{r.MAX_SEVERITY}</span></td>"
        f"<td class='num'><b>{int(r.RISK_SCORE)}</b></td></tr>")

st.markdown(
    f'<div class="card" style="margin-top:14px"><div class="card-h">'
    f'<span class="card-t">Exception worklist</span>'
    f'<span class="card-n">{int(tot["N"]):,} lines match &middot; '
    f'{money(tot["ABS_VAR"])} absolute variance &middot; showing top {len(rows)} '
    f'by risk</span></div>'
    '<table class="tt"><thead><tr>'
    '<th>GL line</th><th>Entity</th><th>Supplier</th><th>Category</th>'
    '<th>Treatment</th><th class="num">Rule</th><th class="num">Variance</th>'
    '<th>Codes</th><th>Severity</th><th class="num">Risk</th>'
    f'</tr></thead><tbody>{body}</tbody></table></div>', unsafe_allow_html=True)

st.markdown(
    f'<div style="color:{MUTED};font-size:.72rem;margin-top:14px;text-align:center">'
    'Every figure traces back to a rule row in <code>TAX.TAX_RULE_DETERMINATION</code> '
    'and a source document reference. Compute stays where the data is; only the '
    'answer travels.</div>', unsafe_allow_html=True)

### Read it like a tax reviewer

1. **Reconciliation panel.** Matched means the computed net GST equals the lodged figure to the cent — not within tolerance, nil. Everything in Investigate is a period where the group filed a number the ledger does not support.
2. **Heatmap.** Hover any cell for the entity, the period and the flagged line count. Read down a column for a bad month across the group, and across a row for an entity with a standing problem.
3. **Worklist.** Drag *Minimum risk score* up to 80. The register collapses to the lines worth a reviewer's morning, and the match count and absolute variance in the panel header move with it because both are recomputed in Snowflake, not filtered in the browser.
4. **Search.** Type a supplier name to pull every flagged line for that vendor across all entities and periods — the question a client asks when a supplier relationship is under review.

> ⚠️ **Green means fewest, not clean**
>
> The heatmap is shaded by rank within the grid, so green is the quietest entity-period *in this population* — every cell still carries a few hundred flagged lines. The distribution is skewed enough that shading by absolute value would put three quarters of the grid in the red and hide the ordering that matters. Worth saying out loud in a client session, because a reviewer who reads green as clean draws exactly the wrong conclusion.

> ✅ **What you did not have to do**
>
> No connection string, no credential, no driver, no `pip install`, no container, no web server, no deployment. `get_active_session()` returns the session you are already authenticated in, so the app inherits your role and therefore your data access — row access and masking policies keep applying. `@st.cache_data(ttl=600)` on every query function means clicking through filters re-renders from cache rather than re-querying, so the warehouse can suspend while people are reading.

### If the app cannot see the objects

If the app loads but every panel reports that an object does not exist, the role running the app is not the role that built the objects. Run the grants below as `ACCOUNTADMIN`, substituting the role your app runs under, then reload the app.

In [ ]:
-- Substitute the role the Streamlit app runs under.
SET app_role = 'SYSADMIN';

GRANT USAGE  ON DATABASE GSI_TAX_WORKSHOP           TO ROLE IDENTIFIER($app_role);
GRANT USAGE  ON SCHEMA   GSI_TAX_WORKSHOP.TAX       TO ROLE IDENTIFIER($app_role);
GRANT USAGE  ON SCHEMA   GSI_TAX_WORKSHOP.CURATED   TO ROLE IDENTIFIER($app_role);
GRANT USAGE  ON SCHEMA   GSI_TAX_WORKSHOP.RAW_DATA  TO ROLE IDENTIFIER($app_role);

GRANT SELECT ON ALL VIEWS           IN SCHEMA GSI_TAX_WORKSHOP.TAX      TO ROLE IDENTIFIER($app_role);
GRANT SELECT ON ALL TABLES          IN SCHEMA GSI_TAX_WORKSHOP.TAX      TO ROLE IDENTIFIER($app_role);
GRANT SELECT ON ALL DYNAMIC TABLES  IN SCHEMA GSI_TAX_WORKSHOP.CURATED  TO ROLE IDENTIFIER($app_role);
GRANT SELECT ON ALL TABLES          IN SCHEMA GSI_TAX_WORKSHOP.RAW_DATA TO ROLE IDENTIFIER($app_role);

GRANT USAGE  ON WAREHOUSE GSI_TAX_WH TO ROLE IDENTIFIER($app_role);

> ✅ **The same two views, now an application**
>
> Live KPI tiles, a reconciliation panel, a risk distribution, an entity heatmap and a filterable worklist — one file, running next to the data, shareable with a URL. No web server, no separate hosting, and no copy of the ledger anywhere.

---

## Step 7 of 8 — The tax team sends you a rate change

*Optional extension — the core flow is complete without this.*

*~10 min · OPTIONAL* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-7)

This is how tax logic actually arrives. Not as a change request, not as a specification, not through a project — as a spreadsheet, by email, from the person who owns the rule. Following the imported-services review in Step 4, the tax team wants the same reverse-charge correction extended to another supply category from the start of next quarter, and they have asked you what it does to the group position *before* it happens.

- Load a tax team spreadsheet into a staging table through the Snowsight interface.
- Clone the entire environment in one statement, at no storage cost.
- Apply the new rules to the clone only, closing off the row they supersede.
- Compare the group GST position across both databases in a single query.
- Throw the scenario away.

### Upload the spreadsheet

No code for this part. The file is `assets/gsi_tax_rule_update.csv` from the workshop bundle — three determination rows, two of them with a `VALID_FROM` in the next quarter.

1. In the left navigation choose **Data**, then **Databases**.
2. Expand `GSI_TAX_WORKSHOP` and select the `TAX` schema.
3. Click **Create**, then **Table**, then **From File**.
4. Drop `gsi_tax_rule_update.csv` into the upload panel.
5. Name the table `STG_RULE_UPDATE`.
6. Snowsight infers the column names and types from the header row — check that `VALID_FROM` and `VALID_TO` came through as `DATE` and that the two boolean columns came through as `BOOLEAN`, then click **Load**.

The tax team's spreadsheet is now a governed table in the `TAX` schema, with inferred types, visible to anyone with access and versioned by the platform. Nobody wrote a loader.

> 💡 **What just changed about that spreadsheet**
>
> It was an attachment in somebody's inbox: no history, no access control, no lineage, and a different copy on four laptops. It is now a table.

### Clone the entire environment

You are about to change tax rules, so do it somewhere that is not production.

In [ ]:
CREATE DATABASE GSI_TAX_SCENARIO CLONE GSI_TAX_WORKSHOP;

In [ ]:
-- Prove the clone is complete: the ledger, the rules and the
-- reconciliation are all there.
SELECT (SELECT COUNT(*) FROM GSI_TAX_SCENARIO.CURATED.GL_HARMONISED)          AS LEDGER_LINES,
       (SELECT COUNT(*) FROM GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION)     AS RULES,
       (SELECT COUNT(*) FROM GSI_TAX_SCENARIO.TAX.V_GST_VS_BAS)               AS ENTITY_PERIODS;

Every schema, every table, every view and all 2,400,000 ledger lines, in one statement. It completes in seconds and consumes no storage, because the clone shares the underlying micro-partitions with the original until one of them diverges. From this point the two databases are independent: anything you do in `GSI_TAX_SCENARIO` is invisible to `GSI_TAX_WORKSHOP`.

### Apply the new rules to the clone only

Note the database qualification on every statement. This is the whole safety mechanism, and it is worth reading twice before you run it.

In [ ]:
-- 0. Fallback, so this notebook runs end to end even if you skipped the
--    upload. Uploading the CSV through the UI is the point of the exercise -
--    that is how a rule change actually arrives - but the staging table has
--    to exist either way. This creates it only if it is missing.
CREATE TABLE IF NOT EXISTS GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE (
    RULE_ID                 NUMBER,
    PRIORITY                NUMBER,
    JURISDICTION            VARCHAR,
    SUPPLY_CATEGORY         VARCHAR,
    SUPPLIER_GST_REGISTERED BOOLEAN,
    IS_CROSS_BORDER         BOOLEAN,
    TAX_TREATMENT           VARCHAR,
    RECOVERABLE             BOOLEAN,
    VALID_FROM              DATE,
    VALID_TO                DATE,
    RULE_DESCRIPTION        VARCHAR
);

INSERT INTO GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE
SELECT * FROM VALUES
    (17,20,'AU','STANDARD_GOODS'   ,FALSE,TRUE,'REVERSE_CHARGE',TRUE,'2026-10-01'::DATE,'9999-12-31'::DATE,'Imported standard goods from an unregistered offshore supplier - reverse charge applies from 1 Oct 2026'),
    (18,20,'AU','CAPITAL_EQUIPMENT',FALSE,TRUE,'REVERSE_CHARGE',TRUE,'2026-10-01'::DATE,'9999-12-31'::DATE,'Imported capital equipment from an unregistered offshore supplier - reverse charge applies from 1 Oct 2026'),
    (19,40,'AU','STANDARD_GOODS'   ,FALSE,TRUE,'TAXABLE'       ,TRUE,'2000-07-01'::DATE,'2026-09-30'::DATE,'Superseded - imported standard goods treated as ordinary taxable supply until 30 Sep 2026')
WHERE NOT EXISTS (SELECT 1 FROM GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE);

In [ ]:
-- 1. Insert the tax team's new determination rows into the CLONE.
INSERT INTO GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION
    (RULE_ID, PRIORITY, JURISDICTION, SUPPLY_CATEGORY, SUPPLIER_GST_REGISTERED,
     IS_CROSS_BORDER, TAX_TREATMENT, RECOVERABLE, VALID_FROM, VALID_TO, RULE_DESCRIPTION)
SELECT RULE_ID, PRIORITY, JURISDICTION, SUPPLY_CATEGORY, SUPPLIER_GST_REGISTERED,
       IS_CROSS_BORDER, TAX_TREATMENT, RECOVERABLE, VALID_FROM, VALID_TO, RULE_DESCRIPTION
FROM GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE;

-- 2. Close off the row the new position supersedes. The existing rule is
--    not deleted and not edited - its VALID_TO is set to the day before
--    the new rule starts, so the historical answer is preserved.
UPDATE GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION
   SET VALID_TO = DATEADD('day', -1, DATE '2026-10-01')
 WHERE RULE_ID = 11
   AND VALID_TO = DATE '9999-12-31';

In [ ]:
-- 3. Read back the rules that now govern imported goods.
SELECT RULE_ID, PRIORITY, SUPPLY_CATEGORY, SUPPLIER_GST_REGISTERED,
       IS_CROSS_BORDER, TAX_TREATMENT, RECOVERABLE, VALID_FROM, VALID_TO
FROM GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION
WHERE SUPPLY_CATEGORY IN ('STANDARD_GOODS', 'CAPITAL_EQUIPMENT')
ORDER BY SUPPLY_CATEGORY, PRIORITY, VALID_FROM;

Read what statement 2 does not do. It does not delete rule 11 and it does not change its treatment. It sets its `VALID_TO`, so a line dated before 1 October 2026 still resolves to the old position and a line dated after it resolves to the new one. That is the difference between a rules table and a configuration file: the historical answer remains reproducible, which is exactly what you need when the auditor asks why a transaction from last year was treated the way it was.

> ⚠️ **No rebuild, no reload, no redeploy**
>
> `TAX.V_GST_DETERMINED` is a view, so the moment those rows land the determined treatment for every affected line has already changed in the clone. There is nothing to refresh, nothing to recompute and nothing to deploy — the next query simply returns a different and correct answer.

### The two positions, side by side

In [ ]:
WITH production AS (
    SELECT ENTITY_NAME,
           SUM(COMPUTED_NET_GST) AS NET_GST
    FROM GSI_TAX_WORKSHOP.TAX.V_GST_VS_BAS
    GROUP BY ENTITY_NAME
),
scenario AS (
    SELECT ENTITY_NAME,
           SUM(COMPUTED_NET_GST) AS NET_GST
    FROM GSI_TAX_SCENARIO.TAX.V_GST_VS_BAS
    GROUP BY ENTITY_NAME
)
SELECT COALESCE(p.ENTITY_NAME, s.ENTITY_NAME)                     AS ENTITY_NAME,
       ROUND(COALESCE(p.NET_GST, 0), 2)                           AS NET_GST_TODAY,
       ROUND(COALESCE(s.NET_GST, 0), 2)                           AS NET_GST_UNDER_NEW_RULES,
       ROUND(COALESCE(s.NET_GST, 0) - COALESCE(p.NET_GST, 0), 2)   AS IMPACT,
       ROUND((COALESCE(s.NET_GST, 0) - COALESCE(p.NET_GST, 0))
             / NULLIF(ABS(COALESCE(p.NET_GST, 0)), 0) * 100, 2)    AS IMPACT_PCT
FROM production p
FULL OUTER JOIN scenario s ON s.ENTITY_NAME = p.ENTITY_NAME
ORDER BY ABS(COALESCE(s.NET_GST, 0) - COALESCE(p.NET_GST, 0)) DESC;

One query, two databases, fourteen entities, and the answer the tax team asked for. Because the new rules carry a future `VALID_FROM`, the impact on *historical* periods is nil by construction — which is itself the finding: no lodged period is restated. The forward effect appears the moment those periods close, and the tax team can see the number now rather than discovering it in a quarterly true-up.

### Throw the scenario away

In [ ]:
DROP DATABASE GSI_TAX_SCENARIO;

Gone. The production database was never touched, and if you decide tomorrow that you needed the scenario after all, `UNDROP DATABASE GSI_TAX_SCENARIO;` brings it back within the retention window.

> ✅ **What the exercise actually cost**
>
> The tax team's spreadsheet became governed, effective-dated reference data. Modelling its impact across a 2,400,000-line ledger cost one clone — no environment request, no data copy, no storage and production untouched.

> 💡 **Every rate change is this exercise**
>
> Every rate change, every treaty update and every legislative amendment is this exercise. A rule arrives from the person who owns it, somebody has to say what it does to the numbers, and the answer has to be produced without disturbing the position already lodged.

---

## Step 8 of 8 — Share the snapshot with the auditor

*Optional extension — the core flow is complete without this.*

*~6 min · OPTIONAL* · [architecture & background](gsi_tax_workshop_lab1_foundation.html#step-8)

The audit snapshot from Step 5 exists and it is immutable. The auditor now needs to read it. The usual answer is an extract — a CSV, a secure file transfer, a copy of the ledger that leaves your control the moment it is sent and is stale before it arrives. There is a better one.

- Create a share and grant it access to the audit snapshot and nothing else.
- Verify exactly what the consumer can see before anybody is added to it.
- Understand why no data is copied and why access remains revocable.

### Create the share and grant it the snapshot only

Read the grant chain carefully. Usage on the database, usage on the `AUDIT` schema, select on one table. There is no grant on `RAW_DATA`, none on `CURATED` and none on `TAX` — so the ledger, the supplier master, the rules and the exception register are all outside the share. The auditor sees the snapshot and nothing else.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE SHARE GST_AUDIT_SNAPSHOT_SHARE
  COMMENT = 'As-lodged GL snapshot for external audit. Read only.';

-- Usage on the container objects. This grants the ability to see the
-- database and the schema exist - it does not grant access to any data.
GRANT USAGE ON DATABASE GSI_TAX_WORKSHOP        TO SHARE GST_AUDIT_SNAPSHOT_SHARE;
GRANT USAGE ON SCHEMA   GSI_TAX_WORKSHOP.AUDIT  TO SHARE GST_AUDIT_SNAPSHOT_SHARE;

-- The only object in the share: the immutable snapshot from Step 5.
-- Nothing in RAW_DATA, CURATED or TAX is granted.
GRANT SELECT ON TABLE GSI_TAX_WORKSHOP.AUDIT.GL_AS_LODGED_20260803
  TO SHARE GST_AUDIT_SNAPSHOT_SHARE;

### Verify what the consumer can actually see

In [ ]:
SHOW GRANTS TO SHARE GST_AUDIT_SNAPSHOT_SHARE;

Three rows: `USAGE` on the database, `USAGE` on the schema, `SELECT` on one table. That output *is* the scope of the share — there is no second place to look and no inherited access to reason about. Run it before you add anybody, and run it again whenever somebody asks what the auditor can see.

> 💡 **No data is copied**
>
> The consumer queries your micro-partitions directly, from their own account, with their own warehouse paying for their own compute. Nothing is extracted, nothing is transferred and there is no second copy to secure, reconcile or later destroy.

The snapshot in this share is a clone taken at a fixed point in time, so its contents will not change — that is deliberate for an audit. The properties of the share itself are what matter: read-only, live against the granted object, revocable in one statement, and precisely scoped to objects you named explicitly.

### Adding the consumer account

One statement completes the share, and it needs something this workshop cannot supply:

> ⚠️ **ALTER SHARE ... ADD ACCOUNTS needs a real account identifier**
>
> The statement is `ALTER SHARE GST_AUDIT_SNAPSHOT_SHARE ADD ACCOUNTS = <organisation>.<account>;` — for example `ADD ACCOUNTS = EXAMPLE_ORG.AUDITOR_ACCOUNT_PLACEHOLDER`, which is an obvious placeholder and will fail. Do not run it with a guessed value.

> ✅ **What you would have done instead**
>
> Exported 2,400,000 rows to a file, agreed a transfer mechanism with the auditor, obtained approval to send client ledger data outside the organisation, and then had no way to withdraw it, no way to tell whether it had been copied again, and no answer when the auditor asked three weeks later whether their extract was still current. A share replaces …

> **What the tax practice can do now**
>
> - [x] see all three ERP extracts side by side, unmodified — *Step 1*
> - [x] show exactly why the group GST position cannot be answered today — *Step 2*
> - [x] query 2,400,000 lines as one governed ledger that maintains itself — *Step 3*
> - [x] read the entire determination logic without reading SQL — *Step 4*
> - [x] reconcile the computed position against what was actually lodged — *Step 5*
> - [x] reproduce any prior filing on demand, and prove nothing moved — *Step 5*